# Short training pilots
CPU preparation first. Four fresh models, 2,000 updates each. Preserve every checkpoint and exact prediction on Drive. Inspect results and bot receipts before extending two survivors to 6,000; full training requires that comparison and a T4 inference check.

In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU first'
from google.colab import drive
drive.mount('/content/drive',timeout_ms=90000)
import hashlib,json
root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
p=root/'prepared/short-pilots-v1/ready.json'
assert hashlib.sha256(p.read_bytes()).hexdigest()=='d0fb09c4f83c44a17247ee643da9db6d6457961b26cb99fbc0818b6c0f8553da'
probe=root/'pilot_reconnect_probe.txt'
probe.write_text('verified from Drive notebook')
assert probe.read_text()=='verified from Drive notebook'
print('DRIVE_NOTEBOOK_CPU_VERIFIED')

Mounted at /content/drive
DRIVE_NOTEBOOK_CPU_VERIFIED


In [ ]:
import os, subprocess, sys
repo=Path('/content/hse-nlp-pilots')
subprocess.run(['git','clone','--depth=1','--branch=2026','https://github.com/OsVox/hse-nlp.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
subprocess.run([sys.executable,'-m','unittest','homeworks.hw2.tests.test_model_sizes','homeworks.hw2.tests.test_continuation'],check=True)
from homeworks.hw2.short_pilots import prepare
prepare()


Export and verify prepared/short-pilots-v1/ready.json from Drive. Only then select a BF16-capable training GPU with at least 40 GiB memory. Do not Run All.

In [ ]:
from google.colab import drive,runtime
try:
    drive.mount('/content/drive',timeout_ms=90000)
    from pathlib import Path
    import hashlib
    p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/short-pilots-v1/launch.py')
    s=p.read_bytes()
    assert hashlib.sha256(s).hexdigest()=='e8a9182c602ecb9f7196bf6e8f6443994c022e3f6918e5ce3d12f5482275dfc9'
    exec(compile(s,str(p),'exec'))
except BaseException:
    runtime.unassign()
    raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PILOTS_STARTED 1315 Automatic notebook-kernel release is armed.


After compute is released, export predictions and evidence from Drive, verify each manifest and exact token decode, submit each checkpoint once to the course bot, and save each receipt to Drive. Selection uses held-out training questions; bot feedback must not become training targets.

In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
import urllib.request,json,hashlib
from pathlib import Path
assert not Path('/dev/nvidia0').exists()
u='https://raw.githubusercontent.com/OsVox/hse-nlp/7945e773d60cf888144fee2a87961a2a3f4ccd1d/homeworks/hw2/hw2_short_pilots_colab.ipynb'
n=json.loads(urllib.request.urlopen(u).read())
s=''.join(n['cells'][4]['source']).encode()
assert hashlib.sha256(s).hexdigest()=='e8a9182c602ecb9f7196bf6e8f6443994c022e3f6918e5ce3d12f5482275dfc9'
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/short-pilots-v1/launch.py')
p.write_bytes(s)
assert p.read_bytes()==s
assert not (p.parent.parent.parent/'runs/short-pilots-v1').exists()
print('LAUNCHER_SAVED_VERIFIED')

LAUNCHER_SAVED_VERIFIED


In [ ]:
from pathlib import Path
import subprocess,json
r=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/short-pilots-v1')
print('PROCESS',pilot_process.poll(),'WATCHDOG',pilot_task.done())
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True))
for p in sorted(r.glob('*/training.log')):print(p.parent.name,p.read_text()[-500:])
print('DRIVER_LOG',(prepared/'execution.log').read_text()[-800:])
if (r/'results.json').exists():
    for x in json.loads((r/'results.json').read_text()):print('RESULT',x['name'],x['validation'],x['numeric']['numeric_correct'],x['numeric']['usable_references'],x['train_and_export_seconds'])
p=r/'gpu-utilization.csv'
if p.exists():
    rows=p.read_text().splitlines()[-60:];print('UTIL_60_MEAN',sum(float(v.split(',')[1]) for v in rows)/len(rows))

PROCESS 0 WATCHDOG True
NVIDIA RTX PRO 6000 Blackwell Server Edition, 0 %, 0 MiB, 97887 MiB

encoder6-lr1e3 th a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=14.42 device=cuda:0 precision=torch.float32 invalid_before_retry=135 beam_repaired=83 handwritten_repairs=0
checkpoint_ready_for_bot=/content/drive/MyDrive/HSE_NLP_HW2/runs/short-pilots-v1/encoder6-lr1e3/step-002000 jsonl_sha256=212df727acd9679bf49cd0d6479e7e44d34eb80ae410d62ae3bde1102a5e597b
finished_steps=2000

encoder6-lr3e4 ith a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=12.80 device=cuda:0 precision=torch.float32 invalid_before_retry=50 beam_repaired=36 handwritten_repairs=0
checkpoint_ready_for_bot=/content/drive/MyDrive/HSE_NLP_HW2/runs/short-pilots-v1/e

In [ ]:
from pathlib import Path
import base64,hashlib,json
r=Path('/content/drive/MyDrive/HSE_NLP_HW2/benchmarks/t4-comparison-v1')
b=(r/'exports.zip').read_bytes();export_payload=base64.b64encode(b).decode();export_offset=0
print('EXPORT',len(b),hashlib.sha256(b).hexdigest(),len(export_payload))
print((r/'execution.json').read_text())

EXPORT 2883561 fd7f959aa36e13ef8c0dd4528a0fb2b818c26a2de900b7c468e7178208bbc5fa 3844748
{
  "elapsed_seconds": 318.569704577,
  "completed_models": 2,
  "gpu": "Tesla T4",
  "training": false
}


In [ ]:
print('EXPORT_CHUNK:'+str(export_offset)+':'+export_payload[export_offset:export_offset+48000]);export_offset+=48000

EXPORT_CHUNK:3840000:0sLjevPZy5rqUIY+5BHo2yHM5W6oKt++xOJ2KhxnTc2977OHtCtex6labb47VcgLX6ddyJo672IXrbBWsbPasXLO6VPYP5+STkhERUhaRtDZ07m+LNNm6A9D36XhafBlenT5ceyJbcMh+P4LL/JQ90Xmy5mcwffNL14S7SUxqYU2bNDE4zIAyEWxSBoylhwbFUHA//VC1he9QLDVEMWkHRfmcFkMhHHALI2hUQz+eGPTf7eaHc+yHeqsyUOr7/S1ok1GACEo5XBKwPJFRZKtMgxKGjjLyu0lG0lIxaxJ8bRVVstvlgQDoiWWE+rsTWcbQRCQlcpYGN3ogll8BsyGmOKKrxnt1e4DUEsBAhQDFAAAAAgAriFQXSI5y5W0PAEAW4IGAB0AAAAAAAAAAAAAAICBAAAAAGJhc2VsaW5lMTgway9mcDMyLWIxMjgtcjAuY3N2UEsBAhQDFAAAAAgAriFQXasbbraqPwEA3/8GAB8AAAAAAAAAAAAAAICB7zwBAGJhc2VsaW5lMTgway9mcDMyLWIxMjgtcjAuanNvbmxQSwECFAMUAAAACACuIVBdXNsk6waZAQBYSxEAKQAAAAAAAAAAAAAAgIHWfAIAYmFzZWxpbmUxODBrL2ZwMzItYjEyOC1yMC5wcm92ZW5hbmNlLmpzb25QSwECFAMUAAAACADEIVBdIjnLlbQ8AQBbggYAHQAAAAAAAAAAAAAAgIEjFgQAYmFzZWxpbmUxODBrL2ZwMzItYjEyOC1yMS5jc3ZQSwECFAMUAAAACADEIVBdqxtutqo/AQDf/wYAHwAAAAAAAAAAAAAAgIESUwUAYmFzZWxpbmUxODBrL2ZwMzItYjEyOC1yMS5qc29ubFBLAQIUAxQAAAAIAMQhUF0jIr/0BZkBAFdLEQApAAAAAAAAAAAAAACAgfmSBgBiYXNlbGluZTE4MGsvZnAzMi1iMTI4LXIxLnByb3ZlbmF

In [ ]:
import zipfile
original_unassign=runtime.unassign
def archive_and_release():
 try:
  r=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/short-pilots-v1')
  with zipfile.ZipFile(r/'final-exports.zip','w',zipfile.ZIP_DEFLATED) as a:
   for p in sorted(r.rglob('*')):
    if p.is_file() and p.suffix in ['.json','.jsonl','.csv','.log']:a.write(p,p.relative_to(r))
   for n in ['execution.complete.json','execution.log']:
    p=prepared/n
    if p.exists():a.write(p,'launcher/'+n)
 finally:original_unassign()
assert pilot_process.poll() is None
runtime.unassign=archive_and_release
print('Final evidence archive will be saved before automatic GPU release.')

Final evidence archive will be saved before automatic GPU release.


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'CPU only'
import os,subprocess,sys
repo=Path('/content/hse-nlp-finalists')
subprocess.run(['git','clone','--depth=1','--branch=2026','https://github.com/OsVox/hse-nlp.git',str(repo)],check=True)
os.chdir(repo)
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()=='10db3cda45ba777e04e4195f003a6b1db5254b24'
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
subprocess.run([sys.executable,'-m','unittest','homeworks.hw2.tests.test_model_sizes','homeworks.hw2.tests.test_continuation'],check=True)
from homeworks.hw2.finalist_pilots import prepare
prepare(['small-lr1e3','encoder6-lr3e4'])

FINALISTS_CPU_READY {"names": ["small-lr1e3", "encoder6-lr3e4"], "parents": {"small-lr1e3": {"files": {"tokenizer.json": "d9fa7b5618323be055ad10ebe760cff2801f251b39dabb7dc62188567c5a6414", "training_state.pt": "fd9d09b2295ba79704761dfd1fbd2213a00673217fb2ca459892e3f2bf96d947", "special_tokens_map.json": "80f4c6c5cf3867d7669ff8aa8f6aebb8a7499322e62d7faae06262d6014edd61", "run_config.json": "fc8b1b5616753d1bf85a5830714570e963696430171fd151e4441710e436c07f", "validation.json": "7b122f12be1c8eca141096169ab296d6c1f6f9bd7bd28f03f8b27cd999bed38c", "model.safetensors": "24581a168e2c3eeb08150c707652a2c88bc9454a3a1f1afb66394842dda4a638", "tokenizer_config.json": "1356e8da73293c9e7bd5a55d12c68de8404b731b8cfc0dbcefbda6a4a0f4c624", "step.json": "d47c24feaffcac7ac05c40282aefa62fed2f67fe587ab3a96d734f5b591cdb55", "config.json": "eead5a90cace8a81e6c7bc3f05f6c25b690028c7a569dcbffb6e08630bd8e9eb", "generation_config.json": "8145d7eecabff8e16a9876617a6d52728e9b8fbe24c426e6bf9ebbd6bfb87737"}, "status": "c

In [ ]:
from pathlib import Path
import json,hashlib,base64
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/finalist-pilots-v1')
n=json.loads(Path('homeworks/hw2/hw2_finalist_pilots_colab.ipynb').read_text())
s=''.join(n['cells'][4]['source']).encode()
assert hashlib.sha256(s).hexdigest()=='a7b712fbd35f6c537f2ea39b946e67d262a1dfac3260e5486cbe64a4a96ec13d'
(p/'launch.py').write_bytes(s)
assert (p/'launch.py').read_bytes()==s
b=(p/'ready.json').read_bytes()
print('READY_HASH',hashlib.sha256(b).hexdigest())
print('READY_B64:'+base64.b64encode(b).decode())

READY_HASH f7150f253bc4e503789020327e1a22dd8256d776b36afbabb6b120b529357022
READY_B64:ewogICJuYW1lcyI6IFsKICAgICJzbWFsbC1scjFlMyIsCiAgICAiZW5jb2RlcjYtbHIzZTQiCiAgXSwKICAicGFyZW50cyI6IHsKICAgICJzbWFsbC1scjFlMyI6IHsKICAgICAgImZpbGVzIjogewogICAgICAgICJ0b2tlbml6ZXIuanNvbiI6ICJkOWZhN2I1NjE4MzIzYmUwNTVhZDEwZWJlNzYwY2ZmMjgwMWYyNTFiMzlkYWJiN2RjNjIxODg1NjdjNWE2NDE0IiwKICAgICAgICAidHJhaW5pbmdfc3RhdGUucHQiOiAiZmQ5ZDA5YjIyOTViYTc5NzA0NzYxZGZkMWZiZDIyMTNhMDA2NzMyMTdmYjJjYTQ1OTg5MmUzZjJiZjk2ZDk0NyIsCiAgICAgICAgInNwZWNpYWxfdG9rZW5zX21hcC5qc29uIjogIjgwZjRjNmM1Y2YzODY3ZDc2NjlmZjhhYThmNmFlYmI4YTc0OTkzMjJlNjJkN2ZhYWUwNjI2MmQ2MDE0ZWRkNjEiLAogICAgICAgICJydW5fY29uZmlnLmpzb24iOiAiZmM4YjFiNTYxNjc1M2QxYmY4NWE1ODMwNzE0NTcwZTk2MzY5NjQzMDE3MWZkMTUxZTQ0NDE3MTBlNDM2YzA3ZiIsCiAgICAgICAgInZhbGlkYXRpb24uanNvbiI6ICI3YjEyMmYxMmJlMWM4ZWNhMTQxMDk2MTY5YWIyOTZkNmMxZjZmOWJkN2JkMjhmMDNmOGIyN2NkOTk5YmVkMzhjIiwKICAgICAgICAibW9kZWwuc2FmZXRlbnNvcnMiOiAiMjQ1ODFhMTY4ZTJjM2VlYjA4MTUwYzcwNzY1MmEyYzg4YmM5NDU0YTNhMWYxYWZiNjYzOTQ4NDJkZG

In [ ]:
import urllib.request,json
from pathlib import Path
root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
u='https://raw.githubusercontent.com/OsVox/hse-nlp/f17ca7b117c7836f97375e996e575a74fcbcd378/homeworks/hw2/'
for x in json.loads((root/'runs/short-pilots-v1/results.json').read_text()):
 name=x['name'];d=root/'runs/short-pilots-v1'/name/'step-002000'
 b=urllib.request.urlopen(u+'submissions/2026-10-16-pilot-'+name+'-step-002000.json').read()
 assert json.loads(b)['jsonl_sha256']==json.loads((d/'results.complete.json').read_text())['jsonl_sha256']
 (d/'bot_receipt.json').write_bytes(b)
 assert (d/'bot_receipt.json').read_bytes()==b
for name in ['report.pdf','hw2_finalist_pilots_colab.ipynb']:
 b=urllib.request.urlopen(u+name).read();(root/name).write_bytes(b);assert (root/name).read_bytes()==b
print('Four bot receipts, report and finalist notebook saved and verified on Drive.')

Four bot receipts, report and finalist notebook saved and verified on Drive.


In [ ]:
import inspect
from google.colab import runtime
print(inspect.getsource(runtime.unassign))

def unassign():
  """Instruct Colab to unassign the currently assigned runtime.

  This function allows a notebook cell to programmatically end the notebook's
  session by unassigning the runtime. This can help users save resources by
  disconnecting soon after execution is finished.

  Raises:
    google.colab.errors.RuntimeManagementError: Error communicating with the
        backend service.
  """
  if not _IS_EXTERNAL_COLAB:
    raise _errors.RuntimeManagementError(
        'This operation is only supported in external Colab.'
    )
  h = _httplib2.Http()
  runtime_server_addr = _os.environ.get('TBE_RUNTIME_ADDR')
  if not runtime_server_addr:
    raise _errors.RuntimeManagementError(
        'Unable to find the runtime management service.'
    )
  resp, _ = h.request(f'http://{runtime_server_addr}/unassign', 'POST')
  if resp.status != _http.HTTPStatus.OK:
    raise _errors.RuntimeManagementError('Unable to request VM unassignment.')
  _output.eval_js('google.colab.kernel.disconne

In [ ]:
from google.colab import drive,runtime
try:
 drive.mount('/content/drive',timeout_ms=90000)
 from pathlib import Path
 import hashlib
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/finalist-pilots-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='a7b712fbd35f6c537f2ea39b946e67d262a1dfac3260e5486cbe64a4a96ec13d'
 exec(compile(s,str(p),'exec'))
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PILOTS_STARTED 3870 Automatic notebook-kernel release is armed.


In [ ]:
from pathlib import Path
import subprocess,json
r=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/finalist-pilots-v1')
print('PROCESS',pilot_process.poll(),'WATCHDOG',pilot_task.done())
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True))
for p in sorted(r.glob('*/training.log')):print(p.parent.name,p.read_text()[-700:])
print('DRIVER_LOG',(prepared/'execution.log').read_text()[-500:])
if (r/'results.json').exists():
 for x in json.loads((r/'results.json').read_text()):print('RESULT',x['name'],x['validation'],x['numeric']['numeric_correct'],x['numeric']['usable_references'],x['train_and_export_seconds'])

PROCESS 0 WATCHDOG True
NVIDIA RTX PRO 6000 Blackwell Server Edition, 0 %, 0 MiB, 97887 MiB

encoder6-lr3e4 oken_loss=0.9782
drive_checkpoint_verified=/content/drive/MyDrive/HSE_NLP_HW2/runs/finalist-pilots-v1/encoder6-lr3e4/step-006000
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=11.05 device=cuda:0 precision=torch.float32 invalid_before_retry=11 beam_repaired=11 handwritten_repairs=0
checkpoint_ready_for_bot=/content/drive/MyDrive/HSE_NLP_HW2/runs/finalist-pilots-v1/encoder6-lr3e4/step-006000 jsonl_sha256=93df0fca9ee39394e874ba6f17c607a7fd6a9a6ab625cb4c8315087c0a42a341
finished_steps=6000

small-lr1e3 _val_token_loss=0.9463
drive_checkpoint_verified=/content/drive/MyDrive/HSE_NLP_HW2/runs/finalist-pilots-v1/small-lr1e3/step-006000
You're using a PreTrainedTokenizerFast tokenize

In [ ]:
from pathlib import Path
import urllib.request,json,hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/finalist-pilots-v1/encoder6-lr3e4/step-006000')
u='https://raw.githubusercontent.com/OsVox/hse-nlp/40fb04c/homeworks/hw2/submissions/2026-10-16-finalist-encoder6-lr3e4-step-006000.json'
b=urllib.request.urlopen(u).read();r=json.loads(b)
assert r['jsonl_sha256']==hashlib.sha256((p/'predictions.jsonl').read_bytes()).hexdigest()
(p/'bot_receipt.json').write_bytes(b)
assert (p/'bot_receipt.json').read_bytes()==b
print('DRIVE_RECEIPT_VERIFIED',r['public_percent'])

DRIVE_RECEIPT_VERIFIED 18.96


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'CPU first'
from google.colab import drive
drive.mount('/content/drive')
import os,subprocess,sys
os.environ['USE_TF']='0';os.environ['USE_FLAX']='0'
repo=Path('/content/hse-nlp-t4-comparison')
subprocess.run(['git','clone','--depth=1','--branch=2026','https://github.com/OsVox/hse-nlp.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
from homeworks.hw2.t4_comparison import prepare
prepare()

Mounted at /content/drive


INFERENCE_CPU_READY {"source_commit": "cf1b9292052b687e842802cee240e5e2a7dad4b8", "inputs_sha256": "b3ad03caf17add1585eb2447dcd574bac5d36ef78d8f3367bb3492d6d81a58ab", "model_sha256": "7d0898f6595ee684df6f8ffa6231bbf914bc81aee91e95298003b1ff69b66deb", "parameters": 42481152, "test_sha256": "8bde59c8d8bb7edc4e8a503be22fc132c4a47773025d333432e78c5489e6522a", "cpu_preflight_passed": true, "model_files": {"tokenizer.json": "d9fa7b5618323be055ad10ebe760cff2801f251b39dabb7dc62188567c5a6414", "special_tokens_map.json": "80f4c6c5cf3867d7669ff8aa8f6aebb8a7499322e62d7faae06262d6014edd61", "run_config.json": "fd46167cf1f6c7ba5ed3f14df4791f59dd005348e2a3cc56599213f2cbbb20f5", "validation.json": "9c07d18f76d9fc7ab8bdc690b33fa0e87c89cb111b338e42c79bf9ef50512c22", "model.safetensors": "7d0898f6595ee684df6f8ffa6231bbf914bc81aee91e95298003b1ff69b66deb", "tokenizer_config.json": "80eb40db7fce330e68a8565c050634eeddcd53f28ca5fd809538bddf0f6204d0", "step.json": "d0b958e6e0b2ee30547e7e346f36a1f6dadd8d0644fe9

In [ ]:
import json,shutil
from homeworks.hw2.checkpoint_io import sha256
from homeworks.hw2.benchmark_production_batches import prepare as prepare_batches
root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
pool=root/'prepared/continuation-v1/probe.jsonl'
assert sha256(pool)==json.loads((root/'prepared/continuation-v1/ready.json').read_text())['pool_sha256']
batch_prepared=root/'prepared/production-batch-finalist6k-v1'
batch_prepared.mkdir(exist_ok=False)
shutil.copyfile(pool,batch_prepared/'probe.jsonl')
assert sha256(batch_prepared/'probe.jsonl')==sha256(pool)
prepare_batches(Path('/content/inference-speed-preparation/model'),batch_prepared/'probe.jsonl',batch_prepared/'ready.json',32768)

BATCH_CPU_READY {"cpu_verified": true, "pool_sha256": "8612f359a5da6123a9bf03f868773bc16d9fc4069e3f16d92b421566e7d6466e", "model_sha256": "7d0898f6595ee684df6f8ffa6231bbf914bc81aee91e95298003b1ff69b66deb", "tokenizer_sha256": "d9fa7b5618323be055ad10ebe760cff2801f251b39dabb7dc62188567c5a6414", "examples": 32768, "batches": [{"batch_size": 64, "examples": 32768, "target_tokens": 2419826}, {"batch_size": 128, "examples": 32768, "target_tokens": 2419826}, {"batch_size": 256, "examples": 32768, "target_tokens": 2419826}, {"batch_size": 512, "examples": 32768, "target_tokens": 2419826}]}


In [ ]:
import json,hashlib,base64
p=root/'prepared/t4-comparison-v1'
s=''.join(json.loads((repo/'homeworks/hw2/hw2_t4_comparison_colab.ipynb').read_text())['cells'][4]['source'])
(p/'launch.py').write_text(s)
assert (p/'launch.py').read_text()==s
print('LAUNCH',hashlib.sha256(s.encode()).hexdigest())
for q in [p/'ready.json',batch_prepared/'ready.json']:
 b=q.read_bytes();print('READY_EXPORT',str(q),hashlib.sha256(b).hexdigest(),base64.b64encode(b).decode())

LAUNCH 9d088e36488dbba4de15af2c16ac980001acabcfb872478bd234af7deedfa0e3
READY_EXPORT /content/drive/MyDrive/HSE_NLP_HW2/prepared/t4-comparison-v1/ready.json 10e6fff1cfd04cda5e08e4487135e755fc7bf9bcbe521c2b0d0b71978b54b494 ewogICJjcHVfdmVyaWZpZWQiOiB0cnVlLAogICJzb3VyY2VfY29tbWl0IjogImNmMWI5MjkyMDUyYjY4N2U4NDI4MDJjZWUyNDBlNWUyYTdkYWQ0YjgiLAogICJzb3VyY2Vfc2hhMjU2IjogImRhZTViNjU0NGMzZDg2NWYyZGVkODMwOWFlN2IyY2JiM2I4OTkzNGQ4NDM5ZDM5YmNhZGI1ZjgzZWFmYTRhZDMiLAogICJtb2RlbHMiOiBbCiAgICB7CiAgICAgICJuYW1lIjogImJhc2VsaW5lMTgwayIsCiAgICAgICJwcmVwYXJlZCI6ICIvY29udGVudC9kcml2ZS9NeURyaXZlL0hTRV9OTFBfSFcyL3ByZXBhcmVkL2luZmVyZW5jZS1zcGVlZC0xODBrLXYxIiwKICAgICAgInJlYWR5IjogewogICAgICAgICJzb3VyY2VfY29tbWl0IjogImViM2Y5NmZiM2JhNTc5MzBhYzZkZDM5ZjI2YTQ4YmFmNGU0ZWY1ODMiLAogICAgICAgICJpbnB1dHNfc2hhMjU2IjogIjkxMGExOGNmYzA0ODVjYjkwYzEyMTQ3YTlmODlkMzk2M2FjYzk2ODQyMDhjN2NkM2U1ZDc3ODE3ZDdiOWU4ZDQiLAogICAgICAgICJtb2RlbF9zaGEyNTYiOiAiMDM5OWE5MjRlZWE2ZWY4NzJhYTBiNDAzYjc3ZmM1ZTMzZDI0OTA5Y2M3NjkwNTM2ODgwYmMwMGIyNTQzNjJjNy

In [ ]:
from google.colab import drive
drive.mount('/content/drive',timeout_ms=90000)
from pathlib import Path
import hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/t4-comparison-v1/launch.py')
s=p.read_text()
assert hashlib.sha256(s.encode()).hexdigest()=='9d088e36488dbba4de15af2c16ac980001acabcfb872478bd234af7deedfa0e3'
exec(compile(s,str(p),'exec'))

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PILOTS_STARTED 1958 Automatic notebook-kernel release is armed.


In [ ]:
from pathlib import Path
import json,subprocess
r=Path('/content/drive/MyDrive/HSE_NLP_HW2/benchmarks/t4-comparison-v1')
print('PROCESS',pilot_process.poll(),'WATCHDOG',pilot_task.done())
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,utilization.gpu,memory.used','--format=csv,noheader'],text=True))
for p in sorted(r.glob('*/results.json')):
 x=json.loads(p.read_text());print(p.parent.name,[(a['batch_size'],a['repeat'],a.get('generation_seconds'),a.get('changed_from_baseline'),a['status']) for a in x['runs']],x.get('completed'))
print('LOG',(prepared/'execution.log').read_text()[-900:])

PROCESS 0 WATCHDOG True
Tesla T4, 0 %, 0 MiB

baseline180k [(64, 0, 29.122432361999984, 0, 'complete'), (128, 0, 16.613615148999997, 0, 'complete'), (256, 0, 10.381593663999979, 0, 'complete'), (256, 1, 10.790551773000004, 0, 'complete'), (128, 1, 16.73783957400002, 0, 'complete'), (64, 1, 28.646765028999937, 0, 'complete')] True
encoder6-6k [(64, 0, 37.58475041700001, 0, 'complete'), (128, 0, 23.502283510000098, 0, 'complete'), (256, 0, 20.051894229000027, 0, 'complete'), (256, 1, 20.55573533200004, 0, 'complete'), (128, 1, 22.78884339000001, 0, 'complete'), (64, 1, 36.479339426000024, 0, 'complete')] True
LOG l": true}, {"batch_size": 128, "median_seconds": 16.67572736150001, "max_seconds": 16.73783957400002, "all_outputs_equal_baseline": true, "repeat_outputs_identical": true}, {"batch_size": 256, "median_seconds": 10.586072718499992, "max_seconds": 10.790551773000004, "all_outputs_equal_baseline": true, "repeat_outputs_identical": true}]}
T4_MODEL_COMPLETE {"name": "encoder6-6k", "

In [ ]:
from pathlib import Path
import json
r=Path('/content/drive/MyDrive/HSE_NLP_HW2/benchmarks/t4-comparison-v1/encoder6-6k/results.json')
x=json.loads(r.read_text());print('HASHES',[(a['batch_size'],a['repeat'],a.get('jsonl_sha256')) for a in x['runs']]);print('SUMMARY',x.get('summary'))

HASHES [(64, 0, '93df0fca9ee39394e874ba6f17c607a7fd6a9a6ab625cb4c8315087c0a42a341'), (128, 0, '93df0fca9ee39394e874ba6f17c607a7fd6a9a6ab625cb4c8315087c0a42a341'), (256, 0, '93df0fca9ee39394e874ba6f17c607a7fd6a9a6ab625cb4c8315087c0a42a341'), (256, 1, '93df0fca9ee39394e874ba6f17c607a7fd6a9a6ab625cb4c8315087c0a42a341'), (128, 1, '93df0fca9ee39394e874ba6f17c607a7fd6a9a6ab625cb4c8315087c0a42a341')]
SUMMARY None


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'CPU first'
from google.colab import drive
drive.mount('/content/drive')
import json,urllib.request,hashlib
u='https://raw.githubusercontent.com/OsVox/hse-nlp/68b3620/homeworks/hw2/hw2_production_batch_colab.ipynb'
b=urllib.request.urlopen(u).read();n=json.loads(b)
exec(''.join(n['cells'][1]['source']))
s=''.join(n['cells'][3]['source'])
(p/'launch.py').write_text(s)
assert (p/'launch.py').read_text()==s
(root/'hw2_production_batch_colab.ipynb').write_bytes(b)
print('BATCH_LAUNCH_READY',hashlib.sha256(s.encode()).hexdigest())

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CPU_BATCH_INPUTS_VERIFIED {'cpu_verified': True, 'pool_sha256': '8612f359a5da6123a9bf03f868773bc16d9fc4069e3f16d92b421566e7d6466e', 'model_sha256': '7d0898f6595ee684df6f8ffa6231bbf914bc81aee91e95298003b1ff69b66deb', 'tokenizer_sha256': 'd9fa7b5618323be055ad10ebe760cff2801f251b39dabb7dc62188567c5a6414', 'examples': 32768, 'batches': [{'batch_size': 64, 'examples': 32768, 'target_tokens': 2419826}, {'batch_size': 128, 'examples': 32768, 'target_tokens': 2419826}, {'batch_size': 256, 'examples': 32768, 'target_tokens': 2419826}, {'batch_size': 512, 'examples': 32768, 'target_tokens': 2419826}]}
BATCH_LAUNCH_READY 5a9cc990c2f882d2dd4515f4396315305658a884cf922c3bd2c42ab931de66c5


In [ ]:
from pathlib import Path
import urllib.request,hashlib
root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
base='https://raw.githubusercontent.com/OsVox/hse-nlp/85532a7/homeworks/hw2/'
for name in ['report.pdf','hw2_inference_colab.ipynb','hw2_grouped_training_colab.ipynb']:
 b=urllib.request.urlopen(base+name).read();p=root/name;p.write_bytes(b);assert p.read_bytes()==b;print('DRIVE_SAVED',name,hashlib.sha256(b).hexdigest())
b=urllib.request.urlopen(base+'experiments/t4-comparison-v1/verification.json').read()
p=root/'benchmarks/t4-comparison-v1/verification.json';p.write_bytes(b);assert p.read_bytes()==b
print('T4_VERIFICATION_SAVED')

DRIVE_SAVED report.pdf 1558bf51051fbd149c686b859912c72bbf1d0e84af4ed8419394236b2f35ac53
DRIVE_SAVED hw2_inference_colab.ipynb b2239389ab7298f9dab3e71f1b567abd7ba645e77e57ee63428461a2af85df0f
DRIVE_SAVED hw2_grouped_training_colab.ipynb a7a3fbebed520cc9701f8a076036798b1f50ae3039deeb1972865da63c8ed221
T4_VERIFICATION_SAVED


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'CPU first'
import json,urllib.request
u='https://raw.githubusercontent.com/OsVox/hse-nlp/85532a7/homeworks/hw2/hw2_grouped_training_colab.ipynb'
n=json.loads(urllib.request.urlopen(u).read())
exec(''.join(n['cells'][2]['source']))

GROUPED_CPU_READY {"cpu_verified": true, "source_commit": "85532a78132c37ceb9cbf3f6cfff5c478d69258f", "source_sha256": "6a73323bc7806ee657b3c7a5f4b6443edaa21b1404f87190e5cc5d5b19eee57f", "parent_sha256": "1afa3ae69473edd930e65aea67766b38929d159fb7fd2ff87dbc4b0d99be4e96", "parent_model_sha256": "7d0898f6595ee684df6f8ffa6231bbf914bc81aee91e95298003b1ff69b66deb", "dataset": {"source_commit": "1a23e45b1138c3dba7c40befeae6d3337153df98", "data_sha256": "a46ee2080a568efb460e0c61cc633354736c1363c184bf9799ef8f2c9c8cd44c", "source_sha256": "e723a8614b3cfd242f5998990e17d89b4d53a022eae0155d427a0712ef05350b", "parquet_files": {"part-00000.parquet": "91831d582faa12d73b1a6b3878e496446cd5df074f17ee4ffea8cc2887c708ec", "part-00001.parquet": "5222d657a1af07a531f1b22503f35741a6547f0831509897d0a279bc8f67d068", "part-00002.parquet": "c6082401ea966aeaa8375018ae1f1d9c7c55cc30e421c45cc933ceacd97dc3de", "part-00003.parquet": "b2b63db46b005e229aa5e91f1010adc763ca9f4b806561b328432fbc9e5c11e9", "part-00004.parque

In [ ]:
from pathlib import Path
import json,hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/grouped-42m-20k-v1')
r=json.loads((p/'ready.json').read_text())
assert r['cpu_verified'] and r['source_commit']=='85532a78132c37ceb9cbf3f6cfff5c478d69258f'
n=json.loads(Path('/content/hse-nlp-grouped-training/homeworks/hw2/hw2_grouped_training_colab.ipynb').read_text())
s=''.join(n['cells'][4]['source'])
(p/'launch.py').write_text(s)
assert (p/'launch.py').read_text()==s
print('LAUNCH_SHA',hashlib.sha256(s.encode()).hexdigest())
print('READY_JSON', (p/'ready.json').read_text())

LAUNCH_SHA e4cb46a686833e5dbb9a244c216e4cb40be58b0536b681f5f417b5676951bf3a
READY_JSON {
  "cpu_verified": true,
  "source_commit": "85532a78132c37ceb9cbf3f6cfff5c478d69258f",
  "source_sha256": "6a73323bc7806ee657b3c7a5f4b6443edaa21b1404f87190e5cc5d5b19eee57f",
  "parent_sha256": "1afa3ae69473edd930e65aea67766b38929d159fb7fd2ff87dbc4b0d99be4e96",
  "parent_model_sha256": "7d0898f6595ee684df6f8ffa6231bbf914bc81aee91e95298003b1ff69b66deb",
  "dataset": {
    "source_commit": "1a23e45b1138c3dba7c40befeae6d3337153df98",
    "data_sha256": "a46ee2080a568efb460e0c61cc633354736c1363c184bf9799ef8f2c9c8cd44c",
    "source_sha256": "e723a8614b3cfd242f5998990e17d89b4d53a022eae0155d427a0712ef05350b",
    "parquet_files": {
      "part-00000.parquet": "91831d582faa12d73b1a6b3878e496446cd5df074f17ee4ffea8cc2887c708ec",
      "part-00001.parquet": "5222d657a1af07a531f1b22503f35741a6547f0831509897d0a279bc8f67d068",
      "part-00002.parquet": "c6082401ea966aeaa8375018ae1f1d9c7c55cc30e421c45cc933ceacd

In [ ]:
from google.colab import drive,runtime
try:
 drive.mount('/content/drive',timeout_ms=90000)
 from pathlib import Path
 import hashlib
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/grouped-42m-20k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='e4cb46a686833e5dbb9a244c216e4cb40be58b0536b681f5f417b5676951bf3a'
 exec(compile(s,str(p),'exec'))
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GROUPED_TRAINING_STARTED 3358 Automatic notebook-kernel release is armed.


In [ ]:
from pathlib import Path
import subprocess,json
print('PROCESS',pilot_process.pid,pilot_process.poll())
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/grouped-42m-20k-v1')
for f in [p/'continuation/training.log',p/'results.json',prepared/'execution.complete.json']:
 if f.exists(): print(str(f),f.read_text()[-750:])
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],text=True))

PROCESS 3358 None
/content/drive/MyDrive/HSE_NLP_HW2/runs/grouped-42m-20k-v1/continuation/training.log tep=18700 train_loss=0.7466 elapsed_s=884.9
step=18750 train_loss=0.5663 elapsed_s=888.6
step=18800 train_loss=0.9179 elapsed_s=892.9
step=18850 train_loss=0.6728 elapsed_s=896.7
step=18900 train_loss=0.4325 elapsed_s=900.6
step=18950 train_loss=0.6843 elapsed_s=904.5
step=19000 train_loss=0.4453 elapsed_s=908.4
step=19000 val_token_loss=0.6254
best_step=19000 best_val_token_loss=0.6254
step=19050 train_loss=0.6993 elapsed_s=912.8
step=19100 train_loss=0.5459 elapsed_s=916.6
step=19150 train_loss=0.7825 elapsed_s=920.8
step=19200 train_loss=0.5710 elapsed_s=924.5
step=19250 train_loss=0.3383 elapsed_s=928.4
step=19300 train_loss=0.6076 elapsed_s=932.4
step=19350 train_loss=0.5165 elapsed_s=936.3
step=19400 train_loss=0.6052 elapsed_s=940.2

98 %, 18821 MiB



In [ ]:
import subprocess
print(subprocess.check_output(['ps','-p',str(pilot_process.pid),'-o','pid,ppid,stat,etime,pcpu,pmem,wchan'],text=True))
from pathlib import Path
for p in [Path('/content/hw2_cache'),Path('/content/grouped-inputs')]:
 print(p,[(f.name,f.stat().st_size) for f in p.glob('*')][-8:])

    PID    PPID STAT     ELAPSED %CPU %MEM WCHAN
   3358    2579 Ssl        00:53 21.2  0.2 hrtimer_nanosleep

/content/hw2_cache [('part-00033.parquet', 30281085), ('part-00004.parquet', 26185069), ('part-00031.parquet', 30272658), ('part-00038.parquet', 23501763), ('part-00026.parquet', 23586227), ('part-00058.parquet', 39321706), ('part-00055.parquet', 39311389), ('part-00009.parquet', 26157219)]
/content/grouped-inputs [('probe.jsonl', 7570890), ('parent', 4096), ('parent.tar', 510474240)]


In [ ]:
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(), 'Select T4 only after grouped training finishes'
from google.colab import drive,runtime
try:
 drive.mount('/content/drive',timeout_ms=90000)
 from pathlib import Path
 import hashlib
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/beam4-180k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='fe255ed35121dbc4132463109996d725efddb3f272274f5e4a8a0bfa05365c3d'
 exec(compile(s,str(p),'exec'))
except BaseException:
 runtime.unassign()
 raise

ValueError: mount failed

In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'CPU only'
from google.colab import drive
import hashlib,json
drive.mount('/content/drive',timeout_ms=180000)
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/beam4-180k-v1')
r=json.loads((p/'ready.json').read_text())
assert hashlib.sha256((p/'source.tar').read_bytes()).hexdigest()==r['source_sha256']
f=p/'cpu_mount_verified.txt'
f.write_text('CPU mount and prepared source verified\n')
assert f.read_text()=='CPU mount and prepared source verified\n'
print('CPU_MOUNT_READY')

In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'CPU preparation only'
from google.colab import drive
drive.mount('/content/drive',force_remount=True,timeout_ms=180000)
import hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/full-42m-bootstrap-v1/prepare.py')
s=p.read_bytes()
assert hashlib.sha256(s).hexdigest()=='edb05571ad0a64ea66f374798c781287c2ad224afbb4ea37b7266322085536c8'
exec(compile(s,str(p),'exec'))

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive,runtime
try:
    import torch
    assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    drive.mount('/content/drive',timeout_ms=180000)
    from pathlib import Path
    import hashlib
    p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/full-42m-100k-v1/launch.py')
    s=p.read_bytes()
    assert hashlib.sha256(s).hexdigest()=='da1706d3040aa7eadce19e9e6267d26c4367e39d7e9f2b8acffb4e4cde2b54f1'
    exec(compile(s,str(p),'exec'))
except BaseException:
    runtime.unassign()
    raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FULL_TRAINING_STARTED 1377 Hard bound 9000 seconds; inspect each saved checkpoint.


In [ ]:
from pathlib import Path
import subprocess
print('PROCESS',full_process.pid,full_process.poll())
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,utilization.gpu,memory.used,memory.total','--format=csv,noheader,nounits'],text=True))
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/full-42m-100k-v1')
for n in ['training.log','execution.json']:
    f=p/n
    print(n,f.read_text()[-1800:] if f.exists() else 'pending')
f=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/full-42m-100k-v1/execution.log')
print('driver',f.read_text()[-1000:])

PROCESS 1377 None
NVIDIA RTX PRO 6000 Blackwell Server Edition, 98, 18953, 97887

training.log arameters=42,481,152
resumed_at_step=20000
new_schedule_start=20000 peak_lr=0.00015
step=20050 train_loss=0.5473 elapsed_s=9.6
step=20100 train_loss=0.3860 elapsed_s=13.8
step=20150 train_loss=0.6018 elapsed_s=17.5
step=20200 train_loss=0.4457 elapsed_s=21.6
step=20250 train_loss=0.7809 elapsed_s=25.4
step=20300 train_loss=0.5371 elapsed_s=29.3
step=20350 train_loss=0.7528 elapsed_s=33.5
step=20400 train_loss=0.5822 elapsed_s=37.2
step=20450 train_loss=0.3525 elapsed_s=41.3
step=20500 train_loss=0.6626 elapsed_s=45.0
step=20550 train_loss=0.4948 elapsed_s=48.9
step=20600 train_loss=0.6835 elapsed_s=52.6
step=20650 train_loss=0.4637 elapsed_s=56.3
step=20700 train_loss=0.7120 elapsed_s=60.4
step=20750 train_loss=0.6060 elapsed_s=64.1
step=20800 train_loss=0.9414 elapsed_s=68.3
step=20850 train_loss=0.6467 elapsed_s=71.8
step=20900 train_loss=0.4756 elapsed_s=75.7
step=20950 train_loss=0.7343 e

In [ ]:
import subprocess
from pathlib import Path
print(subprocess.check_output(['ps','-p',str(full_process.pid),'-o','pid,ppid,stat,etime,pcpu,pmem,wchan'],text=True))
print('driver',Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/full-42m-100k-v1/execution.log').stat().st_size)
print('data',[(str(p),p.stat().st_size) for p in [Path('/content/pilot-data.tar'),Path('/content/full-42m-inputs/parent.tar')] if p.exists()])

    PID    PPID STAT     ELAPSED %CPU %MEM WCHAN
   1377     713 Ssl        00:50 22.2  0.2 hrtimer_nanosleep

driver 0
data [('/content/pilot-data.tar', 1594593280), ('/content/full-42m-inputs/parent.tar', 510484480)]


In [ ]:
from pathlib import Path
import json,re,statistics,time
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/full-42m-100k-v1')
s=(p/'training.log').read_text()
a=re.findall(r'step=(\d+) train_loss=([\d.]+) elapsed_s=([\d.]+)',s)
v=re.findall(r'step=(\d+) val_token_loss=([\d.]+)',s)
g=[r.split(',') for r in (p/'gpu-utilization.csv').read_text().splitlines()[-60:]]
r={'pid':full_process.pid,'returncode':full_process.poll(),'last_training':a[-1] if a else None,'validation':v,'gpu_median_last60':statistics.median(float(x[1]) for x in g),'gpu_max_memory_mib_last60':max(float(x[2]) for x in g),'complete_checkpoints':[f.parent.name for f in p.glob('step-*/results.complete.json')],'observed_utc':time.time()}
print(json.dumps(r))
(p/'live-status.json').write_text(json.dumps(r,indent=2))

{"pid": 1377, "returncode": null, "last_training": ["69700", "0.2886", "4088.6"], "validation": [["21000", "0.6407"], ["22000", "0.6260"], ["23000", "0.6093"], ["24000", "0.5968"], ["25000", "0.5926"], ["26000", "0.5831"], ["27000", "0.5666"], ["28000", "0.5593"], ["29000", "0.5529"], ["30000", "0.5475"], ["31000", "0.5372"], ["32000", "0.5313"], ["33000", "0.5250"], ["34000", "0.5145"], ["35000", "0.5160"], ["36000", "0.5157"], ["37000", "0.5039"], ["38000", "0.4980"], ["39000", "0.4924"], ["40000", "0.4904"], ["41000", "0.4857"], ["42000", "0.4762"], ["43000", "0.4740"], ["44000", "0.4727"], ["45000", "0.4647"], ["46000", "0.4627"], ["47000", "0.4560"], ["48000", "0.4563"], ["49000", "0.4569"], ["50000", "0.4516"], ["51000", "0.4461"], ["52000", "0.4439"], ["53000", "0.4436"], ["54000", "0.4440"], ["55000", "0.4325"], ["56000", "0.4341"], ["57000", "0.4314"], ["58000", "0.4320"], ["59000", "0.4260"], ["60000", "0.4255"], ["61000", "0.4255"], ["62000", "0.4260"], ["63000", "0.4225"], 

2449

In [ ]:
import os
print('CPU_COUNT',os.cpu_count(),'AFFINITY',sorted(os.sched_getaffinity(0)))
print('TRAINING_LIVE',full_process.poll())

CPU_COUNT 48 AFFINITY [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]
TRAINING_LIVE None


In [ ]:
from pathlib import Path
import hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/candidate-vote-180k-v1/launch.py')
s=p.read_bytes()
assert hashlib.sha256(s).hexdigest()=='2f754105de933fe4f21c6823bb7fae300d935a084d3e910e6ed6d7a78a96c6da'
exec(compile(s,str(p),'exec'))

CPU_VOTE_PILOT_STARTED 4034 Main GPU training continues


In [ ]:
from pathlib import Path
import subprocess
print('CPU_PILOT',vote_process.pid,vote_process.poll())
print('GPU_TRAINING',full_process.pid,full_process.poll())
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/benchmarks/candidate-vote-180k-v1')
for name in ['execution.log','pilot/results.json','execution.complete.json']:
    f=p/name
    print(name,f.read_text()[-3000:] if f.exists() else 'pending')
print(subprocess.check_output(['ps','-p',str(vote_process.pid),'-o','pid,stat,etime,pcpu,pmem'],text=True) if vote_process.poll() is None else 'DONE')

CPU_PILOT 4034 0
GPU_TRAINING 1377 None
execution.log {"source_commit": "f047d7d79e58e679866d5571f45d07da0ef8be38", "model_sha256": "0399a924eea6ef872aa0b403b77fc5e33d24909cc7690536880bc00b254362c7", "offset": 512, "rows": 256, "beams": 4, "batch_size": 32, "device": "cpu", "precision": "fp32", "generation_seconds": 33.94163696099997, "selection_seconds": 7.467812804000005, "handwritten_repairs": 0, "training": false, "reference_used_for_selection": false, "policy": "Strict majority of all candidates; otherwise first executable; rank breaks ties", "questions_sha256": "f5a699217f2c816f7189c99bab4321e98baa18445e98d5f8b23a35cd218fd97c", "metrics": {"top_beam": {"rows": 256, "usable_references": 247, "numeric_correct": 141, "numeric_agreement_on_usable": 0.5708502024291497, "prediction_failures": 19, "note": "Agreement with supplied train references; not public pass@1. Incorrect references and restricted execution may affect this diagnostic.", "changed_from_top": 0}, "first_executable": {"

In [ ]:
from pathlib import Path
prepared=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/full-42m-100k-v1')
assert full_task.get_coro().cr_frame.f_globals['prepared']==prepared
print('FULL_WATCHDOG_OUTPUT',prepared)
print('FULL_PROCESS',full_process.pid,full_process.poll())

FULL_WATCHDOG_OUTPUT /content/drive/MyDrive/HSE_NLP_HW2/prepared/full-42m-100k-v1
FULL_PROCESS 1377 None


In [ ]:
from pathlib import Path
import hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/candidate-vote-180k-v1/confirm.py')
s=p.read_bytes()
assert hashlib.sha256(s).hexdigest()=='a3cbe41d9ed42c1aa30ed97a0e2c21046569d60f015a6df6511290f534ce5493'
confirm_ns={'full_process':full_process}
exec(compile(s,str(p),'exec'),confirm_ns)
assert str(full_task.get_coro().cr_frame.f_globals['prepared']).endswith('full-42m-100k-v1')

CPU_VOTE_CONFIRMATION_STARTED 7301 Main GPU training continues


In [ ]:
from pathlib import Path
import json
print('CONFIRMATION',confirm_ns['vote_process'].pid,confirm_ns['vote_process'].poll())
print('GPU_TRAINING',full_process.pid,full_process.poll())
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/benchmarks/candidate-vote-180k-v1')
f=p/'confirmation/results.json'
print(json.dumps(json.loads(f.read_text())['metrics']) if f.exists() else (p/'confirmation.log').read_text()[-500:])

CONFIRMATION 7301 0
GPU_TRAINING 1377 None
{"top_beam": {"rows": 256, "usable_references": 246, "numeric_correct": 136, "numeric_agreement_on_usable": 0.5528455284552846, "prediction_failures": 11, "note": "Agreement with supplied train references; not public pass@1. Incorrect references and restricted execution may affect this diagnostic.", "changed_from_top": 0}, "first_executable": {"rows": 256, "usable_references": 246, "numeric_correct": 136, "numeric_agreement_on_usable": 0.5528455284552846, "prediction_failures": 8, "note": "Agreement with supplied train references; not public pass@1. Incorrect references and restricted execution may affect this diagnostic.", "changed_from_top": 3}, "strict_majority": {"rows": 256, "usable_references": 246, "numeric_correct": 138, "numeric_agreement_on_usable": 0.5609756097560976, "prediction_failures": 8, "note": "Agreement with supplied train references; not public pass@1. Incorrect references and restricted execution may affect this diagnosti

In [ ]:
from pathlib import Path
import hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/candidate-vote-test-180k-v1/launch.py')
s=p.read_bytes()
assert hashlib.sha256(s).hexdigest()=='569c207b75eb9a51dd35470322b9164638c4f2a7e4ffba3d26c7a5bde5c21ac4'
vote_test_ns={'full_process':full_process}
exec(compile(s,str(p),'exec'),vote_test_ns)
assert str(full_task.get_coro().cr_frame.f_globals['prepared']).endswith('full-42m-100k-v1')

CPU_VOTE_TEST_STARTED 11490 GPU training continues


In [ ]:
from pathlib import Path
import subprocess
child=vote_test_ns['vote_process']
print('VOTE_TEST',child.pid,child.poll(),'GPU_TRAINING',full_process.pid,full_process.poll())
print(subprocess.check_output(['ps','-p',str(child.pid),'-o','pid,stat,etime,pcpu,pmem'],text=True) if child.poll() is None else 'DONE')
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/benchmarks/candidate-vote-test-180k-v1')
for n in ['execution.log','execution.complete.json','step-180000/results.complete.json']:
    f=p/n
    print(n,f.read_text()[-2000:] if f.exists() else 'pending')

VOTE_TEST 11490 0 GPU_TRAINING 1377 None
DONE
execution.log You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
{"step": 180000, "source_commit": "124e46d47b5ae740598fb006380ec670b14e2aba", "model_sha256": "0399a924eea6ef872aa0b403b77fc5e33d24909cc7690536880bc00b254362c7", "rows": 1919, "jsonl_sha256": "7094f3c12ad5448f0cbd528a683410d1e2d2db4ec81478cbd4611c7d9ca3b4fb", "generation_seconds": 279.4324081120001, "decode_seconds": 222.9252546780001, "device": "cpu", "num_beams": 4, "batch_size": 64, "selection": "strict_majority_execution", "changed_from_top": 65, "bot_status": "pending", "public_pass_at_1": null, "t4_timing_verified": false, "question_order": "official data/test/test.parquet", "handwritten_repairs": 0}

execution.complete.json {
  "pid": 11490,
  "started_utc": 1792129903.4140542,
  "returncode"

In [ ]:
from pathlib import Path
import hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/hw2-strict-greedy-v1/launch.py')
s=p.read_bytes()
assert hashlib.sha256(s).hexdigest()=='729eb2149c7cad87bf26afc998d47150f2d68ab528f0309c8f7f5a4048aedb99'
greedy_ns={'full_process':full_process}
exec(compile(s,str(p),'exec'),greedy_ns)
assert str(full_task.get_coro().cr_frame.f_globals['prepared']).endswith('full-42m-100k-v1')

CPU_STRICT_GREEDY_STARTED 26780 GPU training continues


In [ ]:
from pathlib import Path
print('STRICT_GREEDY',greedy_ns['greedy_process'].poll(),'TRAINING',full_process.poll())
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/benchmarks/strict-greedy-42m-v1')
for n in ['execution.log','execution.complete.json','step-050000/results.complete.json']:
 f=p/n
 print(n,f.read_text()[-1600:] if f.exists() else 'pending')

STRICT_GREEDY 0 TRAINING None
execution.log Timing is not a T4 result: CUDA is unavailable
`torch_dtype` is deprecated! Use `dtype` instead!
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=99.67 device=cpu precision=torch.float32 invalid_before_retry=3 beam_repaired=0 handwritten_repairs=0
{"step": 50000, "rows": 1919, "jsonl_sha256": "608ae67ffde269773f423ccb37f91439843917054fbfac0bd457cab7a8bc9d7c", "generation_seconds": 99.66642401800027, "source_commit": "8dcb74d2164a30c7b3169dcf7adf76114139761f", "selection": "greedy", "bot_status": "pending", "t4_timing_verified": false}

execution.complete.json {
  "pid": 26780,
  "started_utc": 1792131747.102642,
  "returncode": 0,
  "finished_utc": 1792131850.3179173
}
step-050000/results.complete.json {
  "step": 50000,
  "rows": 1919,
  "

In [ ]:
from pathlib import Path
import hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/hw2-strict-greedy-v1/greedy-60k-launch.py')
s=p.read_bytes()
assert hashlib.sha256(s).hexdigest()=='d3b2d7c5bda2416f1d03043a617659b9c4edeffd0245be713d1847833c0b7f07'
greedy60_ns={'full_process':full_process}
exec(compile(s,str(p),'exec'),greedy60_ns)
assert str(full_task.get_coro().cr_frame.f_globals['prepared']).endswith('full-42m-100k-v1')

CPU_STRICT_GREEDY_STARTED 28555 GPU training continues


In [ ]:
from pathlib import Path
print('STRICT_GREEDY_60K',greedy60_ns['greedy_process'].pid,greedy60_ns['greedy_process'].poll(),'TRAINING',full_process.poll())
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/benchmarks/strict-greedy-42m-60k-v1')
for n in ['execution.log','execution.complete.json','step-060000/results.complete.json']:
 f=p/n
 print(n,f.read_text()[-1600:] if f.exists() else 'pending')

STRICT_GREEDY_60K 28555 0 TRAINING None
execution.log Timing is not a T4 result: CUDA is unavailable
`torch_dtype` is deprecated! Use `dtype` instead!
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=91.84 device=cpu precision=torch.float32 invalid_before_retry=2 beam_repaired=0 handwritten_repairs=0
{"step": 60000, "rows": 1919, "jsonl_sha256": "7b2afa67e33d48e28bb7e48d8bcbff0e42e540b3d8745140eefd643919590106", "generation_seconds": 91.83679603100018, "source_commit": "8dcb74d2164a30c7b3169dcf7adf76114139761f", "selection": "greedy", "bot_status": "pending", "t4_timing_verified": false}

execution.complete.json {
  "pid": 28555,
  "started_utc": 1792132170.0699942,
  "returncode": 0,
  "finished_utc": 1792132265.5268698
}
step-060000/results.complete.json {
  "step": 60000,
  "rows"

In [ ]:
async def stop_at_70k(proc=full_process):
    import asyncio,os,signal,json,time
    from pathlib import Path
    root=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/full-42m-100k-v1')
    while proc.poll() is None:
        p=root/'step-070000/results.complete.json'
        if p.exists() and json.loads(p.read_text())['step']==70000:
            assert json.loads((p.parent/'checkpoint.complete.json').read_text())['status']=='checkpoint_verified'
            (root/'intentional-stop.json').write_text(json.dumps({'reason':'User requested optimizations beyond scaling; stop after durable checkpoint','step':70000,'time_utc':time.time(),'pid':proc.pid}))
            os.killpg(proc.pid,signal.SIGTERM)
            return
        await asyncio.sleep(5)
stop70_task=asyncio.create_task(stop_at_70k())
print('STOP_AT_DURABLE_70K_SCHEDULED; existing watcher releases GPU')

STOP_AT_DURABLE_70K_SCHEDULED; existing watcher releases GPU


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'CPU only'
from google.colab import drive
drive.mount('/content/drive',timeout_ms=180000)
import hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/checkpoint-average-70k-v1/launch.py')
s=p.read_bytes()
assert hashlib.sha256(s).hexdigest()=='c1df70da069745681c2b84768240b4c9683881a4b6838cf146b37ec13952c77b'
average_ns={}
exec(compile(s,str(p),'exec'),average_ns)

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CPU_AVERAGING_STARTED 6186


In [ ]:
from pathlib import Path
import json
print('CPU_ONLY',not Path('/dev/nvidia0').exists(),'PROCESS',average_ns['process'].poll())
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/benchmarks/checkpoint-average-70k-v1')
for stage in ['pilot','confirmation']:
 for f in sorted((p/stage).glob('*.json')):
  r=json.loads(f.read_text());print(stage,f.name,{k:r[k] for k in ['numeric_correct','usable_references','prediction_failures']})
f=p/'selection.json'
print('CONFIRMED',json.loads(f.read_text())['confirmed'] if f.exists() else 'pending')
print('EXPORTS',[str(f.relative_to(p)) for f in p.glob('exports/*/step-*/results.complete.json')])

CPU_ONLY True PROCESS None
pilot avg50_60_70.json {'numeric_correct': 141, 'usable_references': 247, 'prediction_failures': 14}
pilot avg60_70.json {'numeric_correct': 152, 'usable_references': 247, 'prediction_failures': 15}
pilot baseline.json {'numeric_correct': 142, 'usable_references': 247, 'prediction_failures': 17}
confirmation avg60_70.json {'numeric_correct': 136, 'usable_references': 246, 'prediction_failures': 13}
confirmation baseline.json {'numeric_correct': 135, 'usable_references': 246, 'prediction_failures': 10}
CONFIRMED False
EXPORTS ['exports/baseline/step-070000/results.complete.json']


In [ ]:
from pathlib import Path
p=Path('/content/drive/MyDrive/HSE_NLP_HW2')
print((p/'prepared/checkpoint-average-70k-v1/execution.log').read_text()[-2500:])
print([str(f.relative_to(p/'benchmarks/checkpoint-average-70k-v1')) for f in (p/'benchmarks/checkpoint-average-70k-v1').glob('pilot/*')])
for f in (p/'benchmarks/checkpoint-average-70k-v1').glob('pilot/*.log'): print(f.name,f.read_text()[-1200:])

0f98940594", "handwritten_repairs": 0}}, "selected": "avg60_70", "confirmed": false, "confirmation": {"baseline": {"rows": 256, "usable_references": 246, "numeric_correct": 135, "numeric_agreement_on_usable": 0.5487804878048781, "prediction_failures": 10, "note": "Agreement with supplied train references; not public pass@1. Incorrect references and restricted execution may affect this diagnostic.", "model_dir": "/content/drive/MyDrive/HSE_NLP_HW2/runs/full-42m-100k-v1/step-070000", "offset": 768, "beams": 1, "device": "cpu", "precision": "torch.float32", "cpu_threads": 2, "generation_seconds": 47.35708588900002, "batch_size": 64, "max_input_tokens": 128, "max_new_tokens": 192, "model_sha256": "724b4c72914828f6f3facf72c1e7185953cd257df0cb29597f4b3bca5f24c0ac", "evaluator_sha256": "204f4e1aec4a670afa66cca0d397b1fcc8b00666ae286f8c86a4fc29ce22c1ed", "questions_sha256": "c65d773667558d110ce790c0dd21d38dd29df0fa6109dc3fd88ffc4bd7723dd7", "predictions_sha256": "d7165c60a59fca016d7c79cfcddb898

In [ ]:
import subprocess
s=subprocess.run(['ps','-eo','pid,ppid,etimes,pcpu,pmem,args'],capture_output=True,text=True).stdout
print('\n'.join(x for x in s.splitlines() if '-m homeworks.hw2.' in x))

   6186    2192     115  4.2  0.7 /usr/bin/python3 -u -m homeworks.hw2.average_pilot --root /content/drive/MyDrive/HSE_NLP_HW2 --output /content/drive/MyDrive/HSE_NLP_HW2/benchmarks/checkpoint-average-70k-v1
   7231    6186      14  168  2.0 /usr/bin/python3 -u -m homeworks.hw2.numeric_eval --model-dir /content/drive/MyDrive/HSE_NLP_HW2/benchmarks/checkpoint-average-70k-v1/models/avg60_70/step-070000 --cache-dir /content/hw2_cache --output /content/drive/MyDrive/HSE_NLP_HW2/benchmarks/checkpoint-average-70k-v1/pilot/avg60_70.json --offset 512 --rows 256 --beams 1 --cpu-threads 2


In [ ]:
import pyarrow.parquet as pq,json,ast,collections
from pathlib import Path
rows=pq.read_table('/content/hw2_cache/part-00000.parquet',columns=['question','code']).slice(1024).to_pylist()
counts=collections.Counter(r['question'] for r in rows)
sample=rows[::max(1,len(rows)//10000)]
def valid(code):
 try: ast.parse(code); return True
 except SyntaxError:return False
r={'source':'original train shard 0 excluding first 1024 rows','rows':len(rows),'unique_questions':len(counts),'max_question_repetitions':max(counts.values()),'syntax_sample_rows':len(sample),'syntax_failures':sum(not valid(x['code']) for x in sample)}
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/benchmarks/checkpoint-average-70k-v1')
(p/'original-data-audit.json').write_text(json.dumps(r,indent=2))
print(r)

{'source': 'original train shard 0 excluding first 1024 rows', 'rows': 198976, 'unique_questions': 196530, 'max_question_repetitions': 24, 'syntax_sample_rows': 10473, 'syntax_failures': 0}


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'CPU first'
from google.colab import drive
drive.mount('/content/drive',timeout_ms=180000)
import hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/numeric-loss-bootstrap-v1/launch.py')
s=p.read_bytes()
assert hashlib.sha256(s).hexdigest()=='e90c641a2bce156c66386ddcf1d466307408c34c82eae3b7f6ab53e5bb07cb41'
loss_prepare_ns={}
exec(compile(s,str(p),'exec'),loss_prepare_ns)

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NUMERIC_LOSS_CPU_PREPARATION_STARTED 3613


In [ ]:
from pathlib import Path
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/numeric-loss-bootstrap-v1')
print('CPU_ONLY',not Path('/dev/nvidia0').exists(),'PREPARATION',loss_prepare_ns['process'].poll())
print((p/'execution.log').read_text()[-1800:])
print('READY',(p.parent/'numeric-loss-2k-v1/ready.json').exists())

CPU_ONLY True PREPARATION 0
5bf19752e4578", "part-00047.parquet": "fb6cc779bd2ae25cf5329655d4c6c176ae40fb360b110bbf8806969f72856a98", "part-00048.parquet": "1245557b20c3419f576602fbebba464d04187f9b0d4cedc660829b47f20dd955", "part-00049.parquet": "0ec6d7d2b1a6c0e7481284618a11f9df10fea752e7c41b9d73faedf2090f90a0", "part-00050.parquet": "b5a9518358b663512003d62e2f7706add86b63a47969a20e3a1110566869a57e", "part-00051.parquet": "a562a888cfcaafa07a16b54dba79633fc936df34fe61d09303ab11117466d20b", "part-00052.parquet": "e5c605de3a44056b7d5ebaa6442eb3bf2f9898fae29c7560392a65df170f6837", "part-00053.parquet": "e6ccfe88296f320ff4f1d5a881fa8af842ac15d610ad6de8d08fb81ffde0e157", "part-00054.parquet": "82c1eaac80deea7b2ef90e306079b6165a80b011491c2aee5a073da7cf3ef8a4", "part-00055.parquet": "1a793279bf52a90ebbeb290d9c386067b69d28a87e0c31fb89ab90a9d87be751", "part-00056.parquet": "a984ff588e0ab606793ecc0d21b1f29ca39dc03572374aa406c5362a04a7117c", "part-00057.parquet": "27c35d99709bb983e7a072d33031777cd

In [ ]:
from google.colab import drive
drive.mount('/content/drive',timeout_ms=180000)
from pathlib import Path
import hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/numeric-loss-2k-v1/launch.py')
s=p.read_bytes()
assert hashlib.sha256(s).hexdigest()=='310d3bb0acf01a3e292414fd891b305faa6265834b2630b3bb93a53039754902'
loss_gpu_ns={}
exec(compile(s,str(p),'exec'),loss_gpu_ns)

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FULL_TRAINING_STARTED 1792 Hard bound 1800 seconds; two matched 2k arms only.


In [ ]:
from pathlib import Path
import subprocess,json
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/numeric-loss-2k-v1')
print('PROCESS',loss_gpu_ns['full_process'].poll())
print(subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],capture_output=True,text=True).stdout)
for name in ['control','numeric2']:
 f=p/name/'training.log'; print(name,f.read_text()[-400:] if f.exists() else 'pending')
 for stage in ['pilot','confirmation']:
  f=p/name/(stage+'.json')
  if f.exists():
   r=json.loads(f.read_text());print(stage,{k:r[k] for k in ['numeric_correct','usable_references','prediction_failures']})
f=p/'selection.json';print('CONFIRMED',json.loads(f.read_text())['confirmed'] if f.exists() else 'pending')

PROCESS 0
0 %, 3 MiB

control text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=5.16 device=cuda:0 precision=torch.float32 invalid_before_retry=5 beam_repaired=0 handwritten_repairs=0
checkpoint_ready_for_bot=/content/drive/MyDrive/HSE_NLP_HW2/runs/numeric-loss-2k-v1/control/step-072000 jsonl_sha256=00f28602d757f798513a811aad8a61d1454ed03aa3a450c3ca357a6ad20a9acf
finished_steps=72000

pilot {'numeric_correct': 141, 'usable_references': 245, 'prediction_failures': 14}
confirmation {'numeric_correct': 284, 'usable_references': 493, 'prediction_failures': 33}
numeric2 ext followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=4.05 device=cuda:0 precision=torch.float32 invalid_before_retry=2 beam_repaired=0 handwritten_repairs=0
checkpoint_ready_for_bot=/content/drive/MyDrive/HSE_NLP_HW2/runs/numeric-loss-2k-v1/numeric2/step-072000 jsonl_sha256=a74500bb19a7817c29d182f5296c17a48e2fa269f1b84f42c2b5681bc02daf66
finish

In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'CPU first'
from google.colab import drive
drive.mount('/content/drive',timeout_ms=180000)
import hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/broad-screen-bootstrap-v1/launch.py')
s=p.read_bytes()
assert hashlib.sha256(s).hexdigest()=='e736a3d3a7faa03fd0d7d6acb0706ad3cea2118095a6cf2e0eeb73da4172d7e8'
screen_prepare_ns={}
exec(compile(s,str(p),'exec'),screen_prepare_ns)

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BROAD_SCREEN_CPU_PREPARATION_STARTED 3671


In [ ]:
from pathlib import Path
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/broad-screen-bootstrap-v1')
print('CPU_ONLY',not Path('/dev/nvidia0').exists(),'PROCESS',screen_prepare_ns['process'].poll())
print((p/'execution.log').read_text()[-1600:])
print('READY',(p.parent/'broad-screen-1k-v1/ready.json').exists())

CPU_ONLY True PROCESS 0
t-00049.parquet": "0ec6d7d2b1a6c0e7481284618a11f9df10fea752e7c41b9d73faedf2090f90a0", "part-00050.parquet": "b5a9518358b663512003d62e2f7706add86b63a47969a20e3a1110566869a57e", "part-00051.parquet": "a562a888cfcaafa07a16b54dba79633fc936df34fe61d09303ab11117466d20b", "part-00052.parquet": "e5c605de3a44056b7d5ebaa6442eb3bf2f9898fae29c7560392a65df170f6837", "part-00053.parquet": "e6ccfe88296f320ff4f1d5a881fa8af842ac15d610ad6de8d08fb81ffde0e157", "part-00054.parquet": "82c1eaac80deea7b2ef90e306079b6165a80b011491c2aee5a073da7cf3ef8a4", "part-00055.parquet": "1a793279bf52a90ebbeb290d9c386067b69d28a87e0c31fb89ab90a9d87be751", "part-00056.parquet": "a984ff588e0ab606793ecc0d21b1f29ca39dc03572374aa406c5362a04a7117c", "part-00057.parquet": "27c35d99709bb983e7a072d33031777cd154c35ae267b3605b4bdafff56fb9e1", "part-00058.parquet": "6d802692d5ee78e300e8fb94e5b556cc24aa274b622b5806e411edf7e0433673", "part-00059.parquet": "69e1c814c52a97519fc485e7e14f9500cff74585e06c92d648eb2377d

In [ ]:
from google.colab import drive,runtime
try:
 import torch
 assert torch.cuda.is_available(), 'Expected GPU; release incorrect runtime'
 print('GPU',torch.cuda.get_device_name(),flush=True)
 drive.mount('/content/drive',timeout_ms=300000)
 from pathlib import Path
 import hashlib
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/broad-screen-1k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='e1e17c2b78d4a463238bbb22c62707c0b1a76ce3c23c48614de1b73c98444fb6'
 screen_gpu_ns={}
 exec(compile(s,str(p),'exec'),screen_gpu_ns)
except BaseException:
 runtime.unassign()
 raise

GPU NVIDIA RTX PRO 6000 Blackwell Server Edition
Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SHORT_SCREEN_STARTED 3225 Hard bound 2400 seconds; eight matched 1k arms only.


In [ ]:
from pathlib import Path
import json,subprocess
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/broad-screen-1k-v1')
print('PROCESS',screen_gpu_ns['full_process'].poll())
print(subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],capture_output=True,text=True).stdout)
f=p/'progress.json'
r=json.loads(f.read_text()) if f.exists() else {'results':{}}
print('SCORES',{a:{s:(v['numeric_correct'],v['prediction_failures']) for s,v in stages.items()} for a,stages in r['results'].items()})
for d in sorted(p.glob('*/training.log')):
 if d.parent.name not in r['results']:print('ACTIVE',d.parent.name,d.read_text()[-220:])
f=p/'selection.json'
if f.exists():print('RANKING',json.loads(f.read_text())['ranking'])

PROCESS None
43 %, 1429 MiB

SCORES {'control': {'pilot': (137, 11), 'confirmation': (279, 29)}, 'numeric2': {'pilot': (134, 13), 'confirmation': (283, 29)}, 'smooth005': {'pilot': (135, 13), 'confirmation': (290, 31)}, 'equal_question': {'pilot': (134, 12), 'confirmation': (282, 26)}, 'dropout0': {'pilot': (134, 13), 'confirmation': (280, 25)}, 'dropout005': {'pilot': (134, 13), 'confirmation': (279, 27)}, 'lr6e5': {'pilot': (135, 12), 'confirmation': (284, 28)}}
ACTIVE complete_targets pairs=0
checkpoint_ready_for_bot=/content/drive/MyDrive/HSE_NLP_HW2/runs/broad-screen-1k-v1/complete_targets/step-071000 jsonl_sha256=c081d34bab87f4113c678c9e4b820a25845d03e9771007f4b14d95006f0160d2
finished_steps=71000



In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
assert not Path('/dev/nvidia0').exists(), 'CPU only'
drive.mount('/content/drive',timeout_ms=300000)
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/train-diversity-launch.py')
s=p.read_bytes()
assert hashlib.sha256(s).hexdigest()=='ec155decc4b8c7f7d1fbb4623b7b1ae84ff23926227c7a13ac2cf04846a3a212'
exec(compile(s,str(p),'exec'))

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CPU_AUDIT_STARTED 2990 hard bound 900 seconds


In [ ]:
print('CPU_PROCESS',audit_process.poll())
p=audit_root/'progress.json'
print(p.read_text() if p.exists() else 'Restoring and verifying input files')
print((audit_root/'execution.log').read_text()[-1000:])

CPU_PROCESS 0
{"shards_completed": 60, "distinct_question_hashes": 10920597, "elapsed_seconds": 45.65872701600006}
ry": {"count": 122880, "mean": 1.7632242838541667, "median": 1.0, "p90": 2, "p99": 6, "max": 867}, "sample_rows": 122880, "source_tokens": {"count": 122880, "mean": 46.9968017578125, "median": 43.0, "p90": 73, "p99": 106, "max": 262}, "target_tokens": {"count": 122880, "mean": 74.47953287760417, "median": 65.0, "p90": 122, "p99": 223, "max": 503}, "source_over_128": 213, "target_over_192": 2391, "text_after_hash_character_fraction": {"count": 122880, "mean": 0.0462522434506587, "median": 0.0, "p90": 0.21382289416846653, "p99": 0.5255255255255256, "max": 0.8118811881188119}, "token_length_bins": {"0:64": 60233, "64:96": 37446, "96:128": 14560, "128:192": 8205, "192:384": 2426, "384:100000": 10}, "elapsed_seconds": 59.137626139000076, "exclusions": "All copies of the 1024 held-out question texts, across all scanned shards", "limitations": "Exact string groups only. Length an

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
assert not Path('/dev/nvidia0').exists(), 'CPU preparation only'
drive.mount('/content/drive',timeout_ms=300000)
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/batch-order-bootstrap-v1/launch.py')
s=p.read_bytes()
assert hashlib.sha256(s).hexdigest()=='baec11e15f1ed06cb23b0b4a0fbbdcd01b3440de8ebb9fa7da0ec7bc5fb30919'
exec(compile(s,str(p),'exec'))

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BATCH_ORDER_CPU_PREPARATION_STARTED 3071


In [ ]:
print('CPU_PROCESS',process.poll())
print((p/'execution.log').read_text()[-400:])
f=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/batch-order-1k-v1/ready.json')
if f.exists():
 r=json.loads(f.read_text());print('CPU_VERIFIED',r['cpu_verified'],'LAUNCH_SHA',r['launcher_sha256'])

CPU_PROCESS 0
icial data/test/test.parquet; original row order"}, "parent_model_sha256": "724b4c72914828f6f3facf72c1e7185953cd257df0cb29597f4b3bca5f24c0ac", "parent_sha256": "cf9a628276afc4dc95838df11d56400bd83d8319508c5fb031c032f910cc883a", "source_sha256": "1202e3d02489ecb4d88eea8931dc82001ff6b8e64d4932ad58ec0fa524cf20f4", "launcher_sha256": "f40454a602976a50ae0c95ff46da623e45053d949a7fab163b63211a357b1af7"}

CPU_VERIFIED True LAUNCH_SHA f40454a602976a50ae0c95ff46da623e45053d949a7fab163b63211a357b1af7


In [ ]:
from google.colab import drive,runtime
try:
 import torch
 assert torch.cuda.is_available(), 'Expected GPU'
 print('GPU',torch.cuda.get_device_name(),flush=True)
 drive.mount('/content/drive',timeout_ms=300000)
 from pathlib import Path
 import hashlib
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/batch-order-1k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='f40454a602976a50ae0c95ff46da623e45053d949a7fab163b63211a357b1af7'
 batch_gpu_ns={}
 exec(compile(s,str(p),'exec'),batch_gpu_ns)
except BaseException:
 runtime.unassign()
 raise

GPU NVIDIA RTX PRO 6000 Blackwell Server Edition
Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BATCH_ORDER_STARTED 7525 Hard bound 1200 seconds; four matched 1024-update arms only.


In [ ]:
from pathlib import Path
import json,subprocess
b=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/batch-order-1k-v1')
print('PROCESS',batch_gpu_ns['full_process'].poll())
print(subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],capture_output=True,text=True).stdout)
f=b/'progress.json'
r=json.loads(f.read_text()) if f.exists() else {'results':{}}
print('SCORES',{a:{s:(v['numeric_correct'],v['prediction_failures']) for s,v in stages.items()} for a,stages in r['results'].items()})
for d in sorted(b.glob('*/training.log')):
 if d.parent.name not in r['results']:print('ACTIVE',d.parent.name,d.read_text()[-220:])

PROCESS None
0 %, 1825 MiB

SCORES {'control_s42': {'pilot': (137, 12), 'confirmation': (281, 29)}, 'shuffled_s42': {'pilot': (136, 14), 'confirmation': (285, 31)}, 'control_s1042': {'pilot': (138, 16), 'confirmation': (284, 33)}}
ACTIVE shuffled_s1042 _repairs=0
checkpoint_ready_for_bot=/content/drive/MyDrive/HSE_NLP_HW2/runs/batch-order-1k-v1/shuffled_s1042/step-071024 jsonl_sha256=48827d8888ca8826163e5dbc9967774caaa35ed72aba70f5bb6ac0d8b9bebc73
finished_steps=71024



In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'CPU only'
from google.colab import drive
drive.mount('/content/drive',timeout_ms=300000)
import hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/digit-screen-bootstrap-v1/launch.py')
s=p.read_bytes()
assert hashlib.sha256(s).hexdigest()=='58cdaad0a41a7c6406ff4a8256448c66034e472e766faf2354db48303b485bb1'
digit_cpu_ns={}
exec(compile(s,str(p),'exec'),digit_cpu_ns)

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DIGIT_SCREEN_CPU_PREPARATION_STARTED 2200


In [ ]:
from pathlib import Path
import subprocess
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/digit-screen-bootstrap-v1')
print('PROCESS',digit_cpu_ns['process'].poll())
print(subprocess.run(['ps','-p',str(digit_cpu_ns['process'].pid),'-o','etime,time,pcpu,rss,stat'],capture_output=True,text=True).stdout)
print((p/'execution.log').read_text()[-700:])

PROCESS 0
    ELAPSED     TIME %CPU   RSS STAT

 2048, "exact_original_text_roundtrip": true, "unchanged_vocabulary_and_ids": true, "checkpoint_reload_verified": true, "stats": {"original": {"question": {"mean": 46.97283418107833, "median": 43, "truncated": 218}, "code": {"mean": 74.52144048830112, "median": 65, "truncated": 2375}}, "digits": {"question": {"mean": 51.84533875890132, "median": 48, "truncated": 446}, "code": {"mean": 80.94146083418107, "median": 71, "truncated": 3005}}}, "original_tokenizer_sha256": "d9fa7b5618323be055ad10ebe760cff2801f251b39dabb7dc62188567c5a6414", "digit_tokenizer_sha256": "14954be867ea46f5b6175087f1f03c6e23ccc82a71f547d0181c990ab23657d3", "elapsed_seconds": 65.64678022199996, "training_started": false}}



In [ ]:
from google.colab import drive,runtime
try:
 import torch
 assert torch.cuda.is_available(), 'Expected GPU'
 print('GPU',torch.cuda.get_device_name(),flush=True)
 drive.mount('/content/drive',timeout_ms=300000)
 from pathlib import Path
 import hashlib
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/digit-screen-1k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='3e2ee11667d7b6ff8fafb642b83183d75cc28ba73d138433a36f3e3dd888276c'
 digit_gpu_ns={}
 exec(compile(s,str(p),'exec'),digit_gpu_ns)
except BaseException:
 runtime.unassign()
 raise

GPU NVIDIA RTX PRO 6000 Blackwell Server Edition
Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DIGIT_SCREEN_STARTED 7852 Hard bound 900 seconds; two matched 1024-update arms only.


In [ ]:
from pathlib import Path
import json,subprocess
b=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/digit-screen-1k-v1')
print('PROCESS',digit_gpu_ns['full_process'].poll())
print(subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],capture_output=True,text=True).stdout)
f=b/'progress.json'
r=json.loads(f.read_text()) if f.exists() else {'results':{}}
print('SCORES',{a:{s:(v['numeric_correct'],v['prediction_failures']) for s,v in stages.items()} for a,stages in r['results'].items()})
for d in sorted(b.glob('*/training.log')):
 if d.parent.name not in r['results']:print('ACTIVE',d.parent.name,d.read_text()[-220:])

PROCESS 0
0 %, 3 MiB

SCORES {'control': {'pilot': (137, 12), 'confirmation': (281, 29)}, 'digits': {'pilot': (124, 15), 'confirmation': (239, 32)}}


In [ ]:
from google.colab import drive,runtime
try:
 import torch
 assert torch.cuda.is_available(), 'Expected GPU'
 print('GPU',torch.cuda.get_device_name(),flush=True)
 drive.mount('/content/drive',timeout_ms=300000)
 from pathlib import Path
 import hashlib
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/focal-screen-1k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='6b88a95e6b3272303c82fcbdb73746ca899a37a630ae1dd13853c1d1ca3e4aca'
 focal_gpu_ns={}
 exec(compile(s,str(p),'exec'),focal_gpu_ns)
except BaseException:
 runtime.unassign()
 raise

GPU NVIDIA RTX PRO 6000 Blackwell Server Edition
Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FOCAL_SCREEN_STARTED 4925 Hard bound 1200 seconds; three matched 1024-update arms only.


In [ ]:
from pathlib import Path
import json,subprocess
b=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/focal-screen-1k-v1')
print('PROCESS',focal_gpu_ns['full_process'].poll())
print(subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],capture_output=True,text=True).stdout)
f=b/'progress.json'
r=json.loads(f.read_text()) if f.exists() else {'results':{}}
print('SCORES',{a:{s:(v['numeric_correct'],v['prediction_failures']) for s,v in stages.items()} for a,stages in r['results'].items()})
for d in sorted(b.glob('*/training.log')):
 if d.parent.name not in r['results']:print('ACTIVE',d.parent.name,d.read_text()[-220:])

PROCESS None
0 %, 3 MiB

SCORES {'control': {'pilot': (137, 12), 'confirmation': (281, 29)}, 'focal1': {'pilot': (139, 13), 'confirmation': (283, 27)}, 'focal2': {'pilot': (138, 13)}}


In [ ]:
import subprocess
lines=subprocess.run(['ps','-eo','pid,args'],capture_output=True,text=True).stdout.splitlines()
print('\n'.join(x for x in lines if 'homeworks.hw2.train ' in x or ('homeworks.hw2.numeric_eval ' in x and '--score-only' in x)))

   6750 /usr/bin/python3 -u -m homeworks.hw2.train --batch-size 128 --learning-rate 3e-05 --warmup 50 --save-every 71024 --eval-every 71024 --precision bf16 --group-window 32 --mixed-rows-per-shard 4096 --input-pipeline encoded --prefetch-batches 2 --prediction-batch-size 256 --resume-from /content/broad-screen-inputs/parent --steps 71024 --accumulation 1 --seed 42 --cache-dir /content/hw2_cache --output-dir /content/broad-screen-training/focal1 --drive-root /content/drive/MyDrive/HSE_NLP_HW2/runs/focal-screen-1k-v1/focal1 --focal-gamma 1.0 --restart-schedule --predict-at-checkpoint


In [ ]:
from google.colab import drive,runtime
try:
 import torch
 assert torch.cuda.is_available(), 'Expected GPU'
 print('GPU',torch.cuda.get_device_name(),flush=True)
 drive.mount('/content/drive',timeout_ms=300000)
 from pathlib import Path
 import hashlib
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/focal-confirm-1k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='e0c00d0f85e2d0f9fc5a37d90fdd49b98c0fb6b6db63595352b12be1661a5fcb'
 confirm_gpu_ns={}
 exec(compile(s,str(p),'exec'),confirm_gpu_ns)
except BaseException:
 runtime.unassign()
 raise

GPU NVIDIA RTX PRO 6000 Blackwell Server Edition
Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FOCAL_CONFIRM_STARTED 6914 Hard bound 900 seconds; two matched 1024-update arms only.


In [ ]:
from pathlib import Path
import json,subprocess
b=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/focal-confirm-1k-v1')
print('PROCESS',confirm_gpu_ns['full_process'].poll())
print(subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],capture_output=True,text=True).stdout)
f=b/'progress.json'
r=json.loads(f.read_text()) if f.exists() else {'results':{}}
print('SCORES',{a:{s:(v['numeric_correct'],v['prediction_failures']) for s,v in stages.items()} for a,stages in r['results'].items()})
for d in sorted(b.glob('*/training.log')):
 if d.parent.name not in r['results']:print('ACTIVE',d.parent.name,d.read_text()[-220:])

PROCESS None
97 %, 21309 MiB

SCORES {'control': {'pilot': (138, 15)}}
ACTIVE focal1 step=70750 train_loss=0.3081 elapsed_s=65.5
step=70800 train_loss=0.4003 elapsed_s=69.8
step=70850 train_loss=0.3832 elapsed_s=73.5
step=70900 train_loss=0.2627 elapsed_s=77.5
step=70950 train_loss=0.3686 elapsed_s=81.6



In [ ]:
from google.colab import drive,runtime
try:
 from pathlib import Path
 import hashlib
 assert not Path('/dev/nvidia0').exists(), 'CPU only'
 drive.mount('/content/drive',timeout_ms=300000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/cpu-data-tokenizer-audit-v1/cpu-launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='d7819472a082dac271e9f3f5b8a2a85bf71ea6f812df184e7aa509ee5ab746c6'
 cpu_audit_ns={}
 exec(compile(s,str(p),'exec'),cpu_audit_ns)
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CPU_TOKENIZER_AUDIT_STARTED 3025 hard bound 1200 seconds


In [ ]:
from pathlib import Path
import json
b=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/cpu-data-tokenizer-audit-v1')
print('PROCESS',cpu_audit_ns['audit_process'].poll())
for name in ['tokenizers','consistency']:
 p=b/name/'progress.json'
 if p.exists():
  r=json.loads(p.read_text());print(name,{k:v for k,v in r.items() if k!='results'})
p=b/'execution.log'
print(p.read_text()[-1800:] if p.exists() else 'no log yet')

PROCESS 0
tokenizers {'completed': ['original8192', 'balanced8192', 'code8192', 'python8192', 'code16384']}
consistency {'stage': 'execute', 'groups_completed': 503, 'groups_total': 503, 'elapsed_seconds': 83.269416632}
60
SCAN 51 sampled_groups 9056
SCAN 52 sampled_groups 9238
SCAN 53 sampled_groups 9439
SCAN 54 sampled_groups 9629
SCAN 55 sampled_groups 9823
SCAN 56 sampled_groups 9998
SCAN 57 sampled_groups 10164
SCAN 58 sampled_groups 10332
SCAN 59 sampled_groups 10482
SCAN 60 sampled_groups 10510
EXECUTE 50 / 503
EXECUTE 100 / 503
EXECUTE 150 / 503
EXECUTE 200 / 503
EXECUTE 250 / 503
EXECUTE 300 / 503
EXECUTE 350 / 503
EXECUTE 400 / 503
EXECUTE 450 / 503
EXECUTE 500 / 503
RESULT {"purpose": "CPU audit only; original programs unchanged, no training targets generated", "seed": 20261016, "question_sampling_modulus": 1024, "max_groups": 512, "max_programs": 8, "eligible_rows": 11836444, "sampled_rows": 11460, "sampled_question_groups": 10510, "repeated_question_groups": 701, "identica

In [ ]:
from google.colab import drive,runtime
try:
 import torch,hashlib
 from pathlib import Path
 assert torch.cuda.is_available(), 'Expected GPU'
 print('GPU',torch.cuda.get_device_name(),flush=True)
 drive.mount('/content/drive',timeout_ms=300000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/representation-pilot-2k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='63d0aa205de3f45ae3145944feb6b0dbbe2788263278f95b511312e035992b07'
 tokenizer_gpu_ns={}
 exec(compile(s,str(p),'exec'),tokenizer_gpu_ns)
except BaseException:
 runtime.unassign()
 raise

GPU NVIDIA RTX PRO 6000 Blackwell Server Edition
Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TOKENIZER_PILOT_STARTED 7832 Hard bound 1600 seconds; four scratch 2048-update arms only.


In [ ]:
from pathlib import Path
import json,subprocess
b=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/representation-pilot-2k-v1')
print('PROCESS',tokenizer_gpu_ns['full_process'].poll())
print(subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],capture_output=True,text=True).stdout)
f=b/'progress.json'
r=json.loads(f.read_text()) if f.exists() else {'results':{}}
print('SCORES',{a:{s:(v['numeric_correct'],v['prediction_failures']) for s,v in stages.items()} for a,stages in r['results'].items()})
for d in sorted(b.glob('*/training.log')):
 if d.parent.name not in r['results']:print('ACTIVE',d.parent.name,d.read_text()[-250:])

PROCESS None
96 %, 18751 MiB

SCORES {'control': {'pilot': (17, 87), 'confirmation': (28, 156)}, 'code8192': {'pilot': (20, 68), 'confirmation': (36, 122)}, 'code16384': {'pilot': (19, 72)}}
ACTIVE comments n_loss=1.8071 elapsed_s=123.0
step=1350 train_loss=1.3039 elapsed_s=128.0
step=1400 train_loss=1.8565 elapsed_s=131.9
step=1450 train_loss=1.4717 elapsed_s=136.8
step=1500 train_loss=1.9397 elapsed_s=140.9
step=1550 train_loss=1.4732 elapsed_s=145.8



In [ ]:
import sys,json,itertools
sys.path.insert(0,'/content/broad-screen-source')
from homeworks.hw2.data import training_batches
from pathlib import Path
it=training_batches(Path('/content/hw2_cache'),128,seed=42,max_shards=60,mixed_rows_per_shard=4096)
sizes=[len(x) for x in itertools.islice(it,2048)]
r={'updates':len(sizes),'nominal_max_presentations':2048*128,'actual_presentations':sum(sizes),'partial_batches':[n for n in sizes if n!=128],'basis':'Same original stream and complete grouping windows in all four arms; one-to-one cleanup retains row count.'}
(Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/representation-pilot-2k-v1')/'example-budget.json').write_text(json.dumps(r,indent=2))
print(r)

{'updates': 2048, 'nominal_max_presentations': 262144, 'actual_presentations': 262134, 'partial_batches': [118], 'basis': 'Same original stream and complete grouping windows in all four arms; one-to-one cleanup retains row count.'}


In [ ]:
from google.colab import drive,runtime
try:
 import torch,hashlib
 from pathlib import Path
 assert torch.cuda.is_available(), 'Expected GPU'
 print('GPU',torch.cuda.get_device_name(),flush=True)
 drive.mount('/content/drive',timeout_ms=300000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/tokenizer-confirm-2k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='c40a200a14a4e38c334c96376897be60ffd2768ce5be9e35723642e9d739204e'
 tokconfirm_ns={}
 exec(compile(s,str(p),'exec'),tokconfirm_ns)
except BaseException:
 runtime.unassign()
 raise

GPU NVIDIA RTX PRO 6000 Blackwell Server Edition
Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TOKENIZER_CONFIRM_STARTED 3347 Hard bound 900 seconds; two scratch 2048-update arms only.


In [ ]:
from pathlib import Path
import json,subprocess
b=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/tokenizer-confirm-2k-v1')
print('PROCESS',tokconfirm_ns['full_process'].poll())
print(subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],capture_output=True,text=True).stdout)
f=b/'progress.json'
r=json.loads(f.read_text()) if f.exists() else {'results':{}}
print('SCORES',{a:{s:(v['numeric_correct'],v['prediction_failures']) for s,v in stages.items()} for a,stages in r['results'].items()})
for d in sorted(b.glob('*/training.log')):
 if d.parent.name not in r['results']:print('ACTIVE',d.parent.name,d.read_text()[-250:])

PROCESS None
0 %, 823 MiB

SCORES {'control': {'pilot': (18, 65)}}
ACTIVE code8192 device=cuda precision=torch.bfloat16 group_window=32
parameters=42,481,152



In [ ]:
from google.colab import drive,runtime
try:
 from pathlib import Path
 import hashlib
 assert not Path('/dev/nvidia0').exists(), 'CPU only'
 drive.mount('/content/drive',timeout_ms=300000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/frequency-index-cpu-v1/cpu-launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='a9eb829e1f576956d9f64bcd0767a4d077f62f31f002f78227ae8c42df1308ab'
 frequency_ns={}
 exec(compile(s,str(p),'exec'),frequency_ns)
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FREQUENCY_INDEX_CPU_STARTED 3415 hard bound 1200 seconds


In [ ]:
from pathlib import Path
import json
b=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/frequency-index-cpu-v1')
print('PROCESS',frequency_ns['audit_process'].poll())
p=b/'index/progress.json'
print(p.read_text() if p.exists() else 'verifying dataset')
p=b/'execution.log'
print(p.read_text()[-700:] if p.exists() else '')

PROCESS 0
{"stage": "index", "shards": 60}
NDEX 40
INDEX 41
INDEX 42
INDEX 43
INDEX 44
INDEX 45
INDEX 46
INDEX 47
INDEX 48
INDEX 49
INDEX 50
INDEX 51
INDEX 52
INDEX 53
INDEX 54
INDEX 55
INDEX 56
INDEX 57
INDEX 58
INDEX 59
INDEX 60
STREAM 0 {'rows': 131072, 'unique_questions': 130717, 'seconds': 21.415157679999993, 'stream_sha256': '5cf9d0a9ab18fa585f6fdd111a9a0845ea37210d52b1c83600beb92a84449a6d'}
STREAM 0.5 {'rows': 131072, 'unique_questions': 131007, 'seconds': 24.157826949999958, 'stream_sha256': '3966836c5288a8c95304576ae65fcb83e86986af5aa41d0e76fadbce26d852ed'}
STREAM 1 {'rows': 131072, 'unique_questions': 131047, 'seconds': 14.790382431000012, 'stream_sha256': '7db5650541604c44e45b27d42a0cb1db9f58b868ba8a8abee4582e7bf451e868'}



In [ ]:
from google.colab import drive,runtime
try:
 from pathlib import Path
 import hashlib
 assert not Path('/dev/nvidia0').exists(), 'CPU only'
 drive.mount('/content/drive',timeout_ms=300000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/frequency-screen-1k-v1/cpu-launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='22dce5508b2e10d58052326009af8aa6a93ed55ba6c741e960d921cc44bb1ca4'
 freqpre_ns={}
 exec(compile(s,str(p),'exec'),freqpre_ns)
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FREQUENCY_PREFLIGHT_CPU_STARTED 1904 hard bound 1200 seconds


In [ ]:
from pathlib import Path
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/frequency-screen-1k-v1')
print('PROCESS',freqpre_ns['audit_process'].poll())
f=p/'cpu-execution.log'
print(f.read_text()[-1200:] if f.exists() else 'no log')
print('READY',(p/'ready.json').exists())

PROCESS None

READY False


In [ ]:
import subprocess
from pathlib import Path
p=freqpre_ns['audit_process']
print('PROCESS',p.poll())
print(subprocess.run(['ps','-p',str(p.pid),'-o','etime,pcpu,rss,stat'],capture_output=True,text=True).stdout)
for name in ['/content/hw2_cache','/content/frequency-parent-preparation/parent']:
 q=Path(name);print(name,[(f.name,f.stat().st_size) for f in q.glob('*')][-5:])

PROCESS None
    ELAPSED %CPU   RSS STAT
      01:56 36.1 1255488 Rsl

/content/hw2_cache [('part-00007.parquet', 26162520), ('part-00046.parquet', 23553675), ('part-00024.parquet', 23594948), ('part-00028.parquet', 23551367), ('part-00020.parquet', 23581860)]
/content/frequency-parent-preparation/parent [('model.safetensors', 169938176), ('checkpoint.complete.json', 978), ('special_tokens_map.json', 416), ('generation_config.json', 142), ('run_config.json', 32709)]


In [ ]:
from google.colab import drive,runtime
try:
 import torch,hashlib
 from pathlib import Path
 assert torch.cuda.is_available(), 'Expected GPU'
 print('GPU',torch.cuda.get_device_name(),flush=True)
 drive.mount('/content/drive',timeout_ms=300000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/frequency-screen-1k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='71e1efca62b15e1c73302120a89597b8183cf89a4c2de16dafb2bf6a26ccf315'
 freqscreen_ns={}
 exec(compile(s,str(p),'exec'),freqscreen_ns)
except BaseException:
 runtime.unassign()
 raise

In [ ]:
from google.colab import drive,runtime
try:
 import torch,hashlib
 from pathlib import Path
 assert torch.cuda.is_available(), 'Expected GPU'
 print('GPU',torch.cuda.get_device_name(),flush=True)
 drive.mount('/content/drive',timeout_ms=300000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/frequency-screen-1k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='98359e1e9cc6c987f7369aa35bc474d6ad18306314c8efc59533e607881d23ef'
 freqretry_ns={}
 exec(compile(s,str(p),'exec'),freqretry_ns)
except BaseException:
 runtime.unassign()
 raise

GPU NVIDIA RTX PRO 6000 Blackwell Server Edition
Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FREQUENCY_SCREEN_STARTED 7780 Hard bound 1200 seconds; three matched 1024-update arms only.


In [ ]:
from pathlib import Path
import subprocess
print('PROCESS',freqretry_ns['full_process'].poll())
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],text=True))
p=freqretry_ns['local_log'];print('LOCAL_LOG',p.read_text()[-3500:])
r=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/frequency-screen-1k-v1')
for p in sorted(r.glob('*/training.log')):print(p.parent.name,p.read_text()[-600:])

PROCESS None
0 %, 1613 MiB

LOCAL_LOG 
control -screen-1k-v1/control/step-073024
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=5.30 device=cuda:0 precision=torch.float32 invalid_before_retry=4 beam_repaired=0 handwritten_repairs=0
checkpoint_ready_for_bot=/content/drive/MyDrive/HSE_NLP_HW2/runs/frequency-screen-1k-v1/control/step-073024 jsonl_sha256=32153a851016b95bca6653e0cf6e8e6ac046498b6c4585901870331ed0281adf
finished_steps=73024

inverse -screen-1k-v1/inverse/step-073024
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=3.76 device=cuda:0 precision=torch.float32 invalid_befo

In [ ]:
from google.colab import drive,runtime
try:
 import torch,hashlib
 from pathlib import Path
 assert torch.cuda.is_available(), 'Expected GPU'
 print('GPU',torch.cuda.get_device_name(),flush=True)
 drive.mount('/content/drive',timeout_ms=300000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/comment-continuation-1k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='6d9ef80b8947c100c43116763e769ba35b36b1fafa0b7b20ff42c4f6e990d86d'
 comments_ns={}
 exec(compile(s,str(p),'exec'),comments_ns)
except BaseException:
 runtime.unassign()
 raise

GPU NVIDIA RTX PRO 6000 Blackwell Server Edition
Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
COMMENT_CONTINUATION_STARTED 4162 Hard bound 900 seconds; two matched 1024-update arms only.


In [ ]:
from pathlib import Path
import subprocess
print('PROCESS',comments_ns['full_process'].poll())
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],text=True))
p=comments_ns['local_log'];print('LOCAL_LOG',p.read_text()[-2500:])
r=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/comment-continuation-1k-v1')
for p in sorted(r.glob('*/training.log')):print(p.parent.name,p.read_text()[-600:])

PROCESS None
15 %, 25901 MiB

LOCAL_LOG 
comments _loss=0.3831 elapsed_s=28.6
step=72300 train_loss=0.3511 elapsed_s=33.4
step=72350 train_loss=0.3786 elapsed_s=38.5
step=72400 train_loss=0.3446 elapsed_s=42.7
step=72450 train_loss=0.1923 elapsed_s=47.9
step=72500 train_loss=0.3212 elapsed_s=51.8
step=72550 train_loss=0.2486 elapsed_s=56.9
step=72600 train_loss=0.3872 elapsed_s=61.0
step=72650 train_loss=0.2930 elapsed_s=65.9
step=72700 train_loss=0.4074 elapsed_s=70.4
step=72750 train_loss=0.3214 elapsed_s=75.1
step=72800 train_loss=0.4705 elapsed_s=80.3
step=72850 train_loss=0.3404 elapsed_s=84.6
step=72900 train_loss=0.2624 elapsed_s=89.4

control ion-1k-v1/control/step-073024
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=3.84 device=cuda:0 precision=torch.float32 invalid_befor

In [ ]:
from google.colab import drive,runtime
try:
 from pathlib import Path
 import hashlib
 assert not Path('/dev/nvidia0').exists(), 'CPU only'
 drive.mount('/content/drive',timeout_ms=300000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/length-cap-cpu-v1/cpu-launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='f2230854b67d1e6dff0beecc171790fc7973a3e6ac20fd1a181f5aa133173d94'
 length_ns={}
 exec(compile(s,str(p),'exec'),length_ns)
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
LENGTH_CAP_CPU_STARTED 1945 Hard bound 600 seconds; 23 rows only; no GPU or training.


In [ ]:
from pathlib import Path
p=length_ns['full_process']
print('PROCESS',p.pid,p.poll())
log=Path(length_ns['local_log'])
print(log.read_text()[-10000:] if log.exists() else 'No log yet')

NameError: name 'length_ns' is not defined

In [ ]:
from google.colab import drive,runtime
try:
 from pathlib import Path
 import hashlib
 assert not Path('/dev/nvidia0').exists(), 'CPU only'
 drive.mount('/content/drive',timeout_ms=300000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/copy-adapter-cpu-v1/cpu-launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='7d26d94b8f2106ef6c295c3fed5ac4356eb5a9dfc160a70a84aebb3a81bf7f7a'
 copy_ns={}
 exec(compile(s,str(p),'exec'),copy_ns)
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
COPY_ADAPTER_CPU_STARTED 7378 Hard bound 1200 seconds; 545 trainable parameters only; no GPU.


In [ ]:
p=copy_ns['full_process']
print('PROCESS',p.pid,p.poll())
print(copy_ns['local_log'].read_text()[-4000:])

PROCESS 7378 None
FEATURE_ROWS 256 4096
FEATURE_ROWS 512 4096
FEATURE_ROWS 768 4096
FEATURE_ROWS 1024 4096
FEATURE_ROWS 1280 4096
FEATURE_ROWS 1536 4096
FEATURE_ROWS 1792 4096
FEATURE_ROWS 2048 4096
FEATURE_ROWS 2304 4096
FEATURE_ROWS 2560 4096
FEATURE_ROWS 2816 4096
FEATURE_ROWS 3072 4096
FEATURE_ROWS 3328 4096
FEATURE_ROWS 3584 4096
FEATURE_ROWS 3840 4096
FEATURE_ROWS 4096 4096
ADAPTER_STEP {'step': 64, 'loss': 0.4259612262248993}
ADAPTER_STEP {'step': 128, 'loss': 0.4253861904144287}
ADAPTER_STEP {'step': 192, 'loss': 0.4271349310874939}
ADAPTER_STEP {'step': 256, 'loss': 0.4056873321533203}
ADAPTER_STEP {'step': 320, 'loss': 0.39497777819633484}
ADAPTER_STEP {'step': 384, 'loss': 0.45929330587387085}
ADAPTER_STEP {'step': 448, 'loss': 0.43340250849723816}
ADAPTER_STEP {'step': 512, 'loss': 0.3921537399291992}
HELDOUT control pilot 139 13
HELDOUT control confirmation 286 27
HELDOUT candidate pilot 139 13
HELDOUT candidate confirmation 286 27
Timing is not a T4 result: CUDA is unavai

In [ ]:
from google.colab import runtime
try:
 import torch,json
 assert not torch.cuda.is_available()
 print('MUON_RUNTIME_CPU_CHECK',json.dumps({'torch_version':torch.__version__,'native_muon_available':hasattr(torch.optim,'Muon')}),flush=True)
 assert hasattr(torch.optim,'Muon')
finally:
 runtime.unassign()

MUON_RUNTIME_CPU_CHECK {"torch_version": "2.11.0+cpu", "native_muon_available": true}


In [ ]:
from google.colab import drive,runtime
try:
 import torch,hashlib
 from pathlib import Path
 assert torch.cuda.is_available() and hasattr(torch.optim,'Muon')
 drive.mount('/content/drive',timeout_ms=300000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/optimizer-scratch-2k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='b9d0ccde0e504acaf9e47f6986aaafc4110f0d9bf88ce31b133bfce3fc19c0a2'
 optimizer_ns={}
 exec(compile(s,str(p),'exec'),optimizer_ns)
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OPTIMIZER_PILOT_STARTED 2720 Hard bound 1800 seconds; four fresh full-model 2048-update arms only.


In [ ]:
from pathlib import Path
import subprocess
print('PROCESS',optimizer_ns['full_process'].poll())
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],text=True))
print(optimizer_ns['local_log'].read_text()[-3000:])
r=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/optimizer-scratch-2k-v1')
for p in sorted(r.glob('*/training.log')):print(p.parent.name,p.read_text()[-600:])

PROCESS 0
0 %, 3 MiB

OPTIMIZER_THROUGHPUT {"results": {"control": {"seconds_per_step": 0.10375856825001506, "estimated_training_seconds": 212.49754777603084, "input_shape": [128, 103], "target_shape": [128, 185]}, "muon_low": {"seconds_per_step": 0.11496994549999329, "estimated_training_seconds": 235.45844838398625, "input_shape": [128, 103], "target_shape": [128, 185]}}, "torch_version": "2.11.0+cu130", "gpu": "NVIDIA RTX PRO 6000 Blackwell Server Edition", "updates_per_optimizer": 11, "scope": "Discarded timing probe on original train examples; no checkpoint or accuracy result. Fixed-batch throughput is an estimate, not full-run time."}

adamw_high 2k-v1/adamw_high/step-002048
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=4.33 device=cuda:0 precision=torch.float32 invalid_befor

In [ ]:
from google.colab import drive,runtime
try:
 import torch,hashlib
 from pathlib import Path
 assert torch.cuda.is_available() and hasattr(torch.optim,'Muon')
 drive.mount('/content/drive',timeout_ms=300000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/optimizer-confirm-2k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='25f821b037a0e6f1b3c6878f36b4636d087f0c35abd3495f93255e381cacdbdc'
 confirm_ns={}
 exec(compile(s,str(p),'exec'),confirm_ns)
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OPTIMIZER_CONFIRM_STARTED 2904 Hard bound 1000 seconds; two fresh full-model 2048-update arms only.


In [ ]:
from pathlib import Path
import subprocess
print('PROCESS',confirm_ns['full_process'].poll())
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],text=True))
print(confirm_ns['local_log'].read_text()[-2000:])
r=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/optimizer-confirm-2k-v1')
for p in sorted(r.glob('*/training.log')):print(p.parent.name,p.read_text()[-700:])

PROCESS None
0 %, 3 MiB


control _val_token_loss=1.6214
drive_checkpoint_verified=/content/drive/MyDrive/HSE_NLP_HW2/runs/optimizer-confirm-2k-v1/control/step-002048
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=5.42 device=cuda:0 precision=torch.float32 invalid_before_retry=32 beam_repaired=0 handwritten_repairs=0
checkpoint_ready_for_bot=/content/drive/MyDrive/HSE_NLP_HW2/runs/optimizer-confirm-2k-v1/control/step-002048 jsonl_sha256=c27b8d871edc530c104ea1b1c1d9379ed7fb86718a347cd634f5c3182e8ef482
finished_steps=2048

muon_low al_token_loss=1.3338
drive_checkpoint_verified=/content/drive/MyDrive/HSE_NLP_HW2/runs/optimizer-confirm-2k-v1/muon_low/step-002048
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is fas

In [ ]:
from google.colab import drive,runtime
try:
 import torch,hashlib
 from pathlib import Path
 assert torch.cuda.is_available() and hasattr(torch.optim,'Muon')
 drive.mount('/content/drive',timeout_ms=300000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/depth-muon-2k-v1/launch.py')
 s=p.read_bytes()
 assert hashlib.sha256(s).hexdigest()=='197175616e536f5ef03b3a7eb9caad55e0e6dc537fd7064e4cc51f35b7786fbf'
 depth_ns={}
 exec(compile(s,str(p),'exec'),depth_ns)
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DEPTH_PILOT_STARTED 2948 Hard bound 1400 seconds; three fresh depth-balance 2048-update arms only.


In [ ]:
from pathlib import Path
import subprocess
print('PROCESS',depth_ns['full_process'].poll())
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],text=True))
print(depth_ns['local_log'].read_text()[-2000:])
r=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/depth-muon-2k-v1')
for p in sorted(r.glob('*/training.log')):print(p.parent.name,p.read_text()[-700:])

PROCESS None
96 %, 20313 MiB


control step=2048 best_val_token_loss=1.3744
drive_checkpoint_verified=/content/drive/MyDrive/HSE_NLP_HW2/runs/depth-muon-2k-v1/control/step-002048
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=5.35 device=cuda:0 precision=torch.float32 invalid_before_retry=45 beam_repaired=0 handwritten_repairs=0
checkpoint_ready_for_bot=/content/drive/MyDrive/HSE_NLP_HW2/runs/depth-muon-2k-v1/control/step-002048 jsonl_sha256=d463c9d05c32b9f44a6c9bd5d9ab3fac8e1ca5011d061e1b5323e39855e4193e
finished_steps=2048

decoder_heavy 3.0062 elapsed_s=32.4
step=300 train_loss=2.1335 elapsed_s=37.5
step=350 train_loss=2.8159 elapsed_s=43.0
step=400 train_loss=2.1736 elapsed_s=48.2
step=450 train_loss=1.5565 elapsed_s=54.0
step=500 train_loss=2.2954 elapsed_s=59.2
step=550 train

In [ ]:
from google.colab import drive,runtime
try:
 import hashlib
 from pathlib import Path
 drive.mount('/content/drive',timeout_ms=180000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/muon-batch-probe-v1/launch.py')
 payload=p.read_bytes()
 assert hashlib.sha256(payload).hexdigest()=='6f64ae9b5faffeb045f1aa00efbb11af661e99cc6f81465b4ce42da860f0ff8a'
 batch_ns={}
 exec(compile(payload,str(p),'exec'),batch_ns)
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MUON_BATCH_PROBE_STARTED 3926 240-second bound; discarded updates; automatic release.


In [ ]:
print('PROCESS',batch_ns['process'].poll())
print(batch_ns['local_log'].read_text()[-7000:])

PROCESS None
{"repeat": 0, "batch_size": 128, "status": "complete", "seconds": 9.798095587000006, "examples": 8192, "nonpadding_tokens": 900967, "padding_fraction": 0.461906223721428, "examples_per_second": 836.0808411451962, "useful_tokens_per_second": 91953.27724658985, "final_loss": 4.5621256828308105, "peak_allocated_gib": 15.63364028930664, "peak_reserved_gib": 18.439453125}
{"repeat": 0, "batch_size": 256, "status": "complete", "seconds": 7.956238918999986, "examples": 8192, "nonpadding_tokens": 900967, "padding_fraction": 0.48478958516322646, "examples_per_second": 1029.6322274130057, "useful_tokens_per_second": 113240.31482368328, "final_loss": 4.970203876495361, "peak_allocated_gib": 30.817630767822266, "peak_reserved_gib": 37.353515625}



In [ ]:
from google.colab import drive,runtime
try:
 import hashlib
 from pathlib import Path
 drive.mount('/content/drive',timeout_ms=180000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/muon-batch-h100-v1/launch.py')
 payload=p.read_bytes()
 assert hashlib.sha256(payload).hexdigest()=='14ae03b8f23a38c427038ec4961639c6b17cce4399e3fe2dbbd6126412e9d16f'
 batch_ns={}
 exec(compile(payload,str(p),'exec'),batch_ns)
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MUON_BATCH_PROBE_STARTED 4844 240-second bound; discarded updates; automatic release.


In [ ]:
import torch,os,signal,json
from pathlib import Path
actual=torch.cuda.get_device_name()
print('ACTUAL_GPU',actual)
assert 'H100' not in actual
p=batch_ns['process']
if p.poll() is None: os.killpg(p.pid,signal.SIGTERM)
f=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/muon-batch-h100-v1/hardware-mismatch.json')
f.write_text(json.dumps({'requested':'H100','actual':actual,'status':'stopped duplicate fallback; not an H100 benchmark'}))
print('Fallback stopped; existing finalizer saves logs and disconnects.')

ACTUAL_GPU NVIDIA RTX PRO 6000 Blackwell Server Edition
Fallback stopped; existing finalizer saves logs and disconnects.


In [ ]:
from google.colab import drive,runtime
try:
 import torch,hashlib
 from pathlib import Path
 print('ALLOCATED_GPU',torch.cuda.get_device_name())
 assert 'A100' in torch.cuda.get_device_name(), 'Requested GPU unavailable; stop before setup'
 drive.mount('/content/drive',timeout_ms=180000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/muon-batch-a100-v1/launch.py')
 payload=p.read_bytes()
 assert hashlib.sha256(payload).hexdigest()=='66dcb6ad282f8630104abf06fce7b72a647a1c4c76a37ed19abb66952de0f377'
 batch_ns={}
 exec(compile(payload,str(p),'exec'),batch_ns)
except BaseException:
 runtime.unassign()
 raise

ALLOCATED_GPU NVIDIA A100-SXM4-80GB
Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MUON_BATCH_PROBE_STARTED 1371 240-second bound; discarded updates; automatic release.


## Model-size screen: 17.58M / 42.48M / 78.15M
Encoder/decoder tests and their second-seed confirmation are complete: retain 6 encoder / 4 decoder layers. The subsequent CPU prompt-length audit retained source cap 128. Mature-checkpoint Muon switching did not beat the best 575/960 checkpoint.

Next: three fresh scratch Muon models, 2048 updates each, batch 128, seed 42, same data/tokenizer/targets. Validation loss every 512 updates; checkpoint and strict greedy predictions at 2048. Select using both held-out subsets; record public bot scores separately. Equal steps are not equal GPU time. No automatic long training. Prepared files and all outputs are saved to Drive under size-muon-2k-v1. Run only the new launcher; never Run All.

In [ ]:
from google.colab import drive,runtime
try:
 import torch,hashlib,time
 from pathlib import Path
 size_request_utc=time.time()
 print('ALLOCATED_GPU',torch.cuda.get_device_name())
 assert 'RTX PRO 6000' in torch.cuda.get_device_name(), 'Stop on unexpected hardware'
 drive.mount('/content/drive',timeout_ms=180000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/size-muon-2k-v1/launch.py')
 payload=p.read_bytes()
 assert hashlib.sha256(payload).hexdigest()=='3ce82fec05c5daca7b3db64e1d31ceb7bfb9b48168b173a10617e006606f558c'
 size_ns={}
 exec(compile(payload,str(p),'exec'),size_ns)
except BaseException:
 runtime.unassign()
 raise

ALLOCATED_GPU NVIDIA RTX PRO 6000 Blackwell Server Edition
Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SIZE_PILOT_STARTED 5144 Hard bound 1400 seconds; three fresh width/capacity 2048-update arms only.


In [ ]:
# Only run while the original training process is active; never reconnect for monitoring.
from pathlib import Path
import subprocess
print('PROCESS',size_ns['full_process'].poll())
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],text=True))
print(size_ns['local_log'].read_text()[-1800:])
r=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/size-muon-2k-v1')
for p in sorted(r.glob('*/training.log')):
 print(p.parent.name,p.read_text()[-700:])

PROCESS None
97 %, 25023 MiB


control t_step=2048 best_val_token_loss=1.3885
drive_checkpoint_verified=/content/drive/MyDrive/HSE_NLP_HW2/runs/size-muon-2k-v1/control/step-002048
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=4.03 device=cuda:0 precision=torch.float32 invalid_before_retry=53 beam_repaired=0 handwritten_repairs=0
checkpoint_ready_for_bot=/content/drive/MyDrive/HSE_NLP_HW2/runs/size-muon-2k-v1/control/step-002048 jsonl_sha256=74e082deb6a741abba63802352a07a096c4c4877bb65d3392d27c568bee4efe1
finished_steps=2048

large d_s=147.1
step=1150 train_loss=2.0103 elapsed_s=153.9
step=1200 train_loss=1.6865 elapsed_s=159.9
step=1250 train_loss=0.8544 elapsed_s=166.4
step=1300 train_loss=1.7651 elapsed_s=172.8
step=1350 train_loss=1.2203 elapsed_s=179.3
step=1400 train_loss=1.7

In [ ]:
import subprocess,time
from pathlib import Path
print(subprocess.check_output(['ps','-eo','pid,ppid,etime,pcpu,pmem,stat,wchan:24,args'],text=True)[-15000:])
for base in ['/content/broad-screen-source','/content/hw2-cache','/content/drive/MyDrive/HSE_NLP_HW2/runs/size-muon-2k-v1']:
 p=Path(base);print(base,[(x.name,x.stat().st_size) for x in p.glob('*')][:20])
print('DRIVER',size_ns['full_process'].poll(),size_ns['local_log'].read_text()[-2000:])

    PID    PPID     ELAPSED %CPU %MEM STAT WCHAN                    COMMAND
      1       0       19:00  0.0  0.0 Ss   do_sigtimedwait          /sbin/docker-init -- /datalab/run.sh
      7       1       19:00  0.0  0.0 Sl   do_epoll_wait            /tools/node/bin/node /datalab/web/app.js
      9       7       19:00  0.0  0.0 S    do_wait                  /bin/bash -e /usr/local/colab/bin/oom_monitor.sh
     11       1       19:00  0.0  0.0 S    do_wait                  /bin/bash -e /datalab/run.sh
     12      11       19:00  0.0  0.0 Sl   futex_wait_queue         /usr/colab/bin/kernel_manager_proxy --listen_port=6000 --target_port=9000 --logtostderr --listen_host=172.28.0.12 --target_host=172.28.0.12 --tunnel_background_save_url=https://colab.research.google.com/tun/m/cc48301118ce562b961b3c22d803539adc1e0c19/gpu-g4-s-kkb-use5c1-155deq179hz30 --tunnel_background_save_delay=10s --tunnel_periodic_background_save_frequency=30m0s --enable_output_coalescing=true --output_coalescing_require

## Size screen completed: 16 October 2026
All three models trained from scratch for 2048 updates. Public bot scores: 17.58M **90/960**, 42.48M **107/960**, 78.15M **110/960**. Mature best remains **575/960 (59.90%)**, latest reported rank 10/16.

Held-out pilot/confirmation correct: small **27/245 + 49/493**; control **22/245 + 37/493**; large **23/245 + 45/493**. Execution failures: 117 / 219 / 229. Only small passes the predeclared held-out gate. Repeat small versus control with another seed before considering longer training; public scores disagree, so no final optimum is established.

Training process seconds: 137.70 / 202.93 / 283.77. Full driver 729.70 seconds; allocation-to-verified-release upper bound 865.19 seconds. Every final checkpoint and exact greedy output is saved to Drive, downloaded/verified, and scored by the bot. Colab resources verified **0 active sessions, 0 units/hour**. No long run started.

[Drive results](https://drive.google.com/drive/folders/15CJo9B-anNd6iX_2OsTaqFIk2DRm7whu). Training source commit: 926d03d7834ef412e27c4a30bca173a978723c53. Full evidence: experiments/size-muon-2k-v1.

## Full-coverage pipeline check — CPU verified, two short continuations

The old sampler could repeat rows before visiting the whole corpus. CPU verification now covers all 11,836,444 eligible row IDs exactly once, with no held-out rows. Saved coverage counts only optimizer-consumed batches and resumes their exact data cursor. This does not mean a model has trained a full pass.

Run `coverage-pipeline-1k-v1`: two 1,024-update continuations from the same best 42M step-073024 model and AdamW state. Old mixed sampler versus full-coverage traversal. Batch128, BF16, numeric-token weight2, warmup50, LR3e-5; strict greedy decoding. Save both step-074048 checkpoints and predictions to Drive, then export and score in the bot. The worker has a 900-second hard limit and releases the GPU automatically. No long training is authorized by this pilot alone.

CPU preparation: 17 focused tests passed; full-size gradients are finite for both paths. Dataset, parent archive and coverage index were verified before allocation. Source commit: `4ad9e1e`. Do not run the old launch cells or Run All. Use only the launcher immediately below.

In [ ]:
from google.colab import drive,runtime
try:
 import torch,hashlib,time
 from pathlib import Path
 coverage_request_utc=time.time()
 print('ALLOCATED_GPU',torch.cuda.get_device_name())
 assert 'RTX PRO 6000' in torch.cuda.get_device_name(), 'Stop on unexpected hardware'
 drive.mount('/content/drive',timeout_ms=180000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/coverage-pipeline-1k-v1/launch.py')
 payload=p.read_bytes()
 assert hashlib.sha256(payload).hexdigest()=='e3308a8aa26b6999f6c1d8229bd4be3b6e8bf675333436509aa308eb06efa21e'
 coverage_ns={}
 exec(compile(payload,str(p),'exec'),coverage_ns)
except BaseException:
 runtime.unassign()
 raise

ALLOCATED_GPU NVIDIA RTX PRO 6000 Blackwell Server Edition
Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
COVERAGE_PIPELINE_STARTED 2481 Hard bound 900 seconds; two matched 1024-update arms only.


In [ ]:
# Run only while this original worker is active; never reconnect to monitor.
from pathlib import Path
import subprocess
print('PROCESS',coverage_ns['full_process'].poll())
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader'],text=True))
print(coverage_ns['local_log'].read_text()[-1800:])
r=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/coverage-pipeline-1k-v1')
for p in sorted(r.glob('*/training.log')):
 print(p.parent.name,p.read_text()[-700:])

PROCESS None
84 %, 21127 MiB


control 23 elapsed_s=60.8
step=73750 train_loss=0.4248 elapsed_s=64.7
step=73800 train_loss=0.3616 elapsed_s=68.7
step=73850 train_loss=0.4209 elapsed_s=72.9
step=73900 train_loss=0.3748 elapsed_s=76.7
step=73950 train_loss=0.4768 elapsed_s=80.9
step=74000 train_loss=0.4017 elapsed_s=84.7
step=74048 val_token_loss=0.4061
best_step=74048 best_val_token_loss=0.4061
drive_checkpoint_verified=/content/drive/MyDrive/HSE_NLP_HW2/runs/coverage-pipeline-1k-v1/control/step-074048
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



## Coverage pipeline completed — GPU released

Both matched 1,024-update continuations finished. Control: held-out 136/245 and 282/493; public **568/960 (59.17%)**. Corrected coverage: held-out 136/245 and 291/493; public **573/960 (59.69%)**. Previous best stays **575/960 (59.90%)**, rank 10/16 at 20:20 +03 on 2026-10-16. Strict greedy decoding; both checkpoint files and outputs exported from Drive and verified before submission.

Coverage receipt: **131,072 distinct consumed original row IDs, zero repeats, zero full passes**, optimizer step 74,048. The CPU planner covers all 11,836,444 eligible rows exactly once; this does not mean the model has trained a full pass. Saved cursor resumes after consumed batches, not prefetched ones. Prior unique exposure remains unknown.

Driver 276.22 seconds including restoration, training and evaluation; training processes 96.69s control / 96.21s coverage. Whole-driver utilization mean 62.59%, median 96%; peak 21,465 MiB. Allocation and Drive authorization time are additional. Colab active-sessions dialog confirmed **no active sessions** after completion. Do not execute old monitoring cells or Run All.

Engineering gate passed, with five extra held-out execution failures at the permitted boundary. No long training started. Faster attention and SwiGLU remain untested. Full-pass training and exhaustive optimization are not claimed.

[Drive run and checkpoints](https://drive.google.com/drive/folders/1UNJTlNUS98wuP_ozXbzYEM9JGRMd9jWc). Source `4ad9e1e`; bot score messages 788989 and 788994.

## Bounded attention speed probe — CPU checks passed

Compare native T5 attention against an opt-in PyTorch SDPA implementation, preserving T5's unscaled attention, learned relative bias, masking and cache. Six synthetic CPU regression checks passed, plus a full 42.48M-model check on four original training examples: maximum logit difference 6.68e-6, maximum gradient difference 1.37e-7, cached/uncached greedy IDs equal.

Use original 8,192 eligible shard-0 examples; batch128, BF16, Muon .001 + AdamW .0003, three warmup updates, then two reversed-order timing rounds. The profiler records which attention kernels actually run. Require at least 5% median speed improvement before a separate quality pilot. Probe weights are discarded, no checkpoint/score improvement claimed, production defaults unchanged.

Prepared inputs/source were uploaded to Drive and downloaded back before GPU allocation. Source `906b422`. Worker bound 240 seconds, automatic release. Run only the new cell below, never Run All.

In [ ]:
from google.colab import runtime
try:
 from google.colab import drive
 from pathlib import Path
 import hashlib
 drive.mount('/content/drive',timeout_ms=180000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/t5-sdpa-probe-v1/launch.py')
 payload=p.read_bytes()
 assert hashlib.sha256(payload).hexdigest()=='62acb6202f1a11b55cb6b6de120c935f083a8e27d3da5be3b41cc76e3d0a3c6b'
 sdpa_ns={}
 exec(compile(payload,str(p),'exec'),sdpa_ns)
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
T5_SDPA_PROBE_STARTED 4222 240-second bound; discarded updates; automatic release.


## Attention probe completed — retain native T5, GPU off

The bounded probe finished in **29.35 seconds**, excluding allocation, Drive mounting and installation. Native attention: **1,486.39 examples/s** median; SDPA: **1,356.30 examples/s** (**8.75% slower**). Both reversed rounds agree. Native peak allocated memory ~15.64 GiB; SDPA ~16.20 GiB. Profiles contain both efficient-attention and ordinary math kernels.

The proposed implementation fails the predeclared 5% speed-gain gate. Keep the current attention implementation; no quality training pilot for this version. This does not rule out every fused-attention approach. CPU equivalence tests passed, but no broad GPU quality equivalence is claimed. Probe weights were discarded, no checkpoint or bot submission was produced.

Results were saved to Drive, then exported and verified. **Zero active sessions and zero units/hour** confirmed in Colab after completion. [Drive speed evidence](https://drive.google.com/drive/folders/1s-_hsVw5WCQXKebuSObFsdw1priJbH0D). Source `906b422`. The full-corpus token audit continues locally on CPU; no long training has started.

## Whole-corpus token audit completed — no GPU needed

Scanned **all 11,836,444 eligible original rows across 60 verified shards** without truncation. Mean question/program lengths: **46.58 / 73.76 tokens**; medians **43 / 64**; maxima **741 / 1,208**. No unknown tokens or literal pad/internal end tokens were found.

| Input / output limits | Questions cut | Programs cut | Rows with any cut |
|---|---:|---:|---:|
| 128 / 192 (current) | 19,870 | 212,045 | 228,035 (1.927%) |
| 128 / 384 | 19,870 | 1,184 | 20,930 (0.177%) |
| 192 / 384 | 390 | 1,184 | 1,551 (0.013%) |
| 256 / 512 | 62 | 153 | 213 (0.0018%) |

In an equal-sized sample of 32 over-limit targets per shard (1,920 programs), all original programs parse. After the current truncation, **472 no longer parse; 1,344 lose all return statements; only 104 remain parseable with a return**. This conditional sample is not a population-weighted estimate and parsing is not a correctness score. No programs were executed or rewritten.

The CPU scan took **700.30 seconds**. Per-row arrays, hashes, summaries and source were [saved to Drive and downloaded back](https://drive.google.com/drive/folders/1ZL_83v5t_wJGfYVxvTAxzBe9SSwmeP4W). Next: isolate a larger output-limit comparison before changing input limits, then continue the remaining short architecture tests. Keep greedy decoding and exact row tracking. Rare outliers still need an explicit treatment before claiming every token is seen.

**This verifies the corpus, not a completed training pass. No long training started. GPU remains off. Best public score remains 575/960 (59.90%); no new submission was produced by these audits.**

## Target length: bounded matched comparison

CPU checks passed before GPU allocation: 11 focused tests, full-size parent forward/backward, and 4,096 original rows with identical source tokens, row order and batch membership. 80 targets retain additional text; padded target positions rise from 720,896 to 912,512 in this sample.

Compare **training target caps 192 and 384**, each starting from the same own 42.48M checkpoint at step 73,024 and training for **1,024 updates**. Batch128, AdamW moments, learning rate3e-5, warmup50, numeric-token weight2, BF16 and seed42 remain matched. Full-coverage traversal in both arms; grouping still uses capped target length192. Input cap128, validation cap192 and **greedy generation cap192** remain unchanged.

Require positive executed-answer gains on both held-out subsets, no additional total failures and no decline against the parent. Repeat any survivor before combining. Public feedback is recorded separately. Both retained checkpoints and outputs go to Drive before export and bot evaluation. Hard worker bound900 seconds; automatic GPU release. **No long run and no completed full-pass claim.** Source `698b8bf`.

In [ ]:
from google.colab import runtime
try:
 from google.colab import drive
 from pathlib import Path
 import hashlib
 drive.mount('/content/drive',timeout_ms=180000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/target-length-1k-v1/launch.py')
 payload=p.read_bytes()
 assert hashlib.sha256(payload).hexdigest()=='9cdfe6a4ed1d357d6c1f4bfa8cfafaf79a82b3eac373cb87b8e6640801226369'
 target_ns={}
 exec(compile(payload,str(p),'exec'),target_ns)
except BaseException:
 runtime.unassign()
 raise

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TARGET_LENGTH_STARTED 2266 Hard bound 900 seconds; two matched 1024-update arms only.


In [ ]:
# Read-only monitoring while the bounded worker is active; do not rerun after shutdown.
import json,subprocess
from pathlib import Path
print('WORKER_RETURN_CODE', target_ns['full_process'].poll())
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,power.draw','--format=csv,noheader'],text=True))
for p in sorted(Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/target-length-1k-v1').glob('*/training.log')):
 print(p.parent.name, p.read_text()[-1200:])

WORKER_RETURN_CODE None
100 %, 44873 MiB, 366.62 W

control 194, 2174, 2151, 2207, 2205, 2191, 2153, 2218, 2160, 2190, 2117, 2210, 2182], "completed_full_passes": 0, "unique_rows_since_tracking_started": 131072, "coverage_percent": 1.1073596090177085, "presentations": 131072, "repeated_presentations_since_tracking_started": 0, "historical_coverage_before_start_step": "unknown", "start_step": 73024, "source_length": 128, "target_length": 192, "grouping_target_length": 192, "note": "Original row IDs, not distinct question texts; row coverage does not prove untruncated token coverage."}
drive_checkpoint_verified=/content/drive/MyDrive/HSE_NLP_HW2/runs/target-length-1k-v1/control/step-074048
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=4.43 device=cuda:0 precision=torch.float32 inval

## Target-length comparison completed — GPU off

Both 1,024-update arms finished successfully. Each consumed **131,072 unique original rows**, with zero repeats and the **same row-order hash**. This is **1.11% of eligible rows, not a full training pass**.

| Training target limit | Held-out /245 | Held-out /493 | Execution failures | Bot public /960 |
|---|---:|---:|---:|---:|
| 192 (control) | 135 | 291 | 45 | **573 (59.69%)** |
| 384 | 138 | 284 | 48 | **570 (59.38%)** |

The longer limit fails the predeclared held-out gate: +3 and −7 correct answers, plus three execution failures. It is **not selected for combination or a long run**. This short continuation does not prove that complete targets have no long-term benefit. Input cap 128 and greedy generation cap 192 were fixed in both arms.

The whole driver took **296.55 seconds (4.94 minutes)**, excluding allocation, sign-in and installation. Training-process times: **98.28 versus 116.56 seconds**. Steady logged throughput: **1,567 versus 1,270 examples/s**. Whole-driver GPU utilization: median 96%, mean 65.69%; peak memory 46,411 MiB. A live training snapshot showed 100% utilization. **No active Colab sessions; zero units/hour** verified after automatic shutdown.

Both checkpoints and exact 1,919-row outputs were saved to Drive, then exported and checksum-verified before bot evaluation. Bot score messages: 789035 and 789039. Best remains **575/960 (59.90%), rank 10/16**, verified by the bot at 20:53 (+03), 16 October 2026. [Drive experiment and checkpoints](https://drive.google.com/drive/folders/1oY5wnHlSfvObtrdUaU_GUNZPOur2Alxm).

Before scaling: address remaining long-sequence truncation, finish the prioritized SwiGLU comparison, and require a completed consumed-row full-pass record. **No claim that all optimizations are exhausted.** Source `698b8bf`; evidence `experiments/target-length-1k-v1`.

## SwiGLU versus gated-GELU — bounded scratch comparison

CPU checks passed before GPU allocation: **10 focused tests**, identical full random weights, exact parameter count **42,481,152**, finite gradients and one finite Muon update for each full-size model. No preflight weights are retained.

Two fresh **2,048-update** runs compare the current gated-GELU feed-forward layers with **SwiGLU**. Keep 6 encoder / 4 decoder layers, width512, feed-forward width1536, batch128, BF16, seed42, warmup200, Muon0.001 and AdamW0.0003 fixed. Both use the original tokenizer and original targets, full-coverage traversal, source/target caps128/192 and strict greedy decoding. This isolates activation choice; existing long-sequence truncation remains unresolved.

Require identical full initialization and consumed-row hashes, **262,144 unique rows per arm**, positive executed-answer gains on both held-out subsets, and no extra total execution failures. Repeat a survivor with another seed before combining. Public feedback does not select the recipe.

Save both final checkpoints and all1,919 outputs to Drive before export and bot evaluation. Record loss every512 steps, but retain only the final checkpoint per arm. Worker hard limit1,000 seconds; automatic GPU release. **No long run or completed full-pass claim.** Source `2735e1e`. [Research motivation](https://arxiv.org/abs/2002.05202).

In [ ]:
from google.colab import runtime
try:
 from google.colab import drive
 from pathlib import Path
 import hashlib
 drive.mount('/content/drive',timeout_ms=180000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/activation-muon-2k-v1/launch.py')
 payload=p.read_bytes()
 assert hashlib.sha256(payload).hexdigest()=='df3d47a1d3482d01d57552786f22fc97dc3176da7b95baf91f7f86bf91a26018'
 activation_ns={}
 exec(compile(payload,str(p),'exec'),activation_ns)
except BaseException:
 runtime.unassign()
 raise

## Launch stopped before training — CPU-only recovery

The first allocation failed at `drive.mount` with `ValueError: mount failed`, before the verified launcher or training worker started. No activation-run folder or checkpoint exists. Colab automatically released the GPU; zero active sessions and zero units/hour were verified. The browser also reported expired sign-in/output JavaScript and was refreshed; the existing MCP connection was confirmed again.

Do not rerun the failed training cell blindly. The next cell runs only on CPU, verifies the mounted prepared source and shared archives, records a small Drive read/write receipt, then releases its CPU runtime. No training or generation is performed.

In [ ]:
from google.colab import runtime
from pathlib import Path
try:
 assert not Path('/dev/nvidia0').exists(), 'CPU-only Drive recovery'
 from google.colab import drive
 import json,hashlib,time
 drive.mount('/content/drive',timeout_ms=180000)
 prepared=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/activation-muon-2k-v1')
 ready=json.loads((prepared/'ready.json').read_text())
 assert ready['cpu_verified']
 checks={}
 for name,key in [('source.tar','source_sha256'),('launch.py','launcher_sha256')]:
  digest=hashlib.sha256((prepared/name).read_bytes()).hexdigest()
  assert digest==ready[key]
  checks[name]=digest
 index=Path(ready['coverage_index_archive_path'])
 assert hashlib.sha256(index.read_bytes()).hexdigest()==ready['coverage_index_sha256']
 dataset=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/data.tar')
 assert dataset.is_file() and dataset.stat().st_size==1594593280
 assert not Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/activation-muon-2k-v1').exists()
 record=dict(cpu_only=True,drive_read_write_verified=True,checks=checks,index_verified=True,dataset_archive_bytes=dataset.stat().st_size,training_started=False,checked_utc=time.time())
 out=prepared/'cpu-drive-recovery.json'
 out.write_text(json.dumps(record,indent=2))
 assert json.loads(out.read_text())==record
 print('CPU_DRIVE_RECOVERY_VERIFIED',json.dumps(record))
finally:
 runtime.unassign()

## CPU recovery after replacing the disconnected bridge

The stale bridge could not reconnect. A fresh official MCP adapter connected to this same Drive notebook in the corporate account and successfully read the saved cells. CPU with standard RAM is verified; no runtime is active. This separate recovery phase verifies prepared Drive artifacts and writes a durable read/write receipt before any GPU allocation. Earlier failed training and recovery cells remain unchanged; no training process or checkpoint exists.

In [1]:
from google.colab import runtime
from pathlib import Path
try:
 assert not Path('/dev/nvidia0').exists(), 'CPU-only recovery'
 from google.colab import drive
 import json, hashlib, time
 print('CPU_ONLY_DRIVE_VERIFICATION_START', flush=True)
 drive.mount('/content/drive', timeout_ms=180000)
 prepared=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/activation-muon-2k-v1')
 ready=json.loads((prepared/'ready.json').read_text())
 assert ready['cpu_verified']
 checks={}
 for name,key in [('source.tar','source_sha256'),('launch.py','launcher_sha256')]:
  digest=hashlib.sha256((prepared/name).read_bytes()).hexdigest()
  assert digest==ready[key]
  checks[name]=digest
 index=Path(ready['coverage_index_archive_path'])
 assert hashlib.sha256(index.read_bytes()).hexdigest()==ready['coverage_index_sha256']
 dataset=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/data.tar')
 assert dataset.is_file() and dataset.stat().st_size==1594593280
 assert not Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/activation-muon-2k-v1').exists()
 record=dict(cpu_only=True,drive_read_write_verified=True,checks=checks,index_verified=True,dataset_archive_bytes=dataset.stat().st_size,training_started=False,checked_utc=time.time(),recovery='fresh MCP bridge and browser tab')
 out=prepared/'cpu-drive-recovery.json'
 out.write_text(json.dumps(record,indent=2))
 assert json.loads(out.read_text())==record
 print('CPU_DRIVE_RECOVERY_VERIFIED',json.dumps(record),flush=True)
finally:
 runtime.unassign()


CPU_ONLY_DRIVE_VERIFICATION_START
Mounted at /content/drive
CPU_DRIVE_RECOVERY_VERIFIED {"cpu_only": true, "drive_read_write_verified": true, "checks": {"source.tar": "eaceaadc025e619f257d9db112628acf456ed3fbf94d2891b5535df17094e65c", "launch.py": "df3d47a1d3482d01d57552786f22fc97dc3176da7b95baf91f7f86bf91a26018"}, "index_verified": true, "dataset_archive_bytes": 1594593280, "training_started": false, "checked_utc": 1792179376.167914, "recovery": "fresh MCP bridge and browser tab"}


## Verified recovery: bounded activation comparison

CPU cell DubI5i2dNhtg mounted Drive, verified prepared source/launcher/index hashes, wrote cpu-drive-recovery.json, and released its runtime. That receipt was exported from Drive. This guarded GPU phase requires the verified receipt and no prior execution result, then launches the immutable 2735e1e activation experiment. Both variants retain the same 128/192 limits to isolate activation choice. The new untruncated path remains a separate pending GPU probe. Two 2,048-update arms only; hard worker bound 1,000 seconds and automatic runtime release. No long training or model scaling.

In [1]:
from google.colab import runtime
try:
 from google.colab import drive
 from pathlib import Path
 import json, hashlib
 drive.mount('/content/drive',timeout_ms=180000)
 prepared=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/activation-muon-2k-v1')
 recovered=json.loads((prepared/'cpu-drive-recovery.json').read_text())
 assert recovered['cpu_only'] and recovered['drive_read_write_verified'] and recovered['index_verified']
 assert not (prepared/'execution.complete.json').exists(), 'Read prior execution before any repeat'
 assert not Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/activation-muon-2k-v1').exists(), 'Run already exists'
 p=prepared/'launch.py'
 payload=p.read_bytes()
 assert hashlib.sha256(payload).hexdigest()=='df3d47a1d3482d01d57552786f22fc97dc3176da7b95baf91f7f86bf91a26018'
 activation_recovered_ns={}
 exec(compile(payload,str(p),'exec'),activation_recovered_ns)
except BaseException:
 runtime.unassign()
 raise


Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ACTIVATION_PILOT_STARTED 1583 Hard bound 1000 seconds; two fresh activation 2048-update arms only.


In [7]:
from pathlib import Path
import subprocess,json
print('PROCESS',activation_recovered_ns['full_process'].poll())
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,power.draw','--format=csv,noheader'],text=True))
p=activation_recovered_ns['local_log']; print(p.read_text()[-1800:])
r=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/activation-muon-2k-v1')
for p in sorted(r.glob('*/training.log')): print(p.parent.name,p.read_text()[-1100:])

PROCESS None
0 %, 15023 MiB, 156.01 W


control s": 0, "unique_rows_since_tracking_started": 262144, "coverage_percent": 2.214719218035417, "presentations": 262144, "repeated_presentations_since_tracking_started": 0, "historical_coverage_before_start_step": "unknown", "start_step": 0, "source_length": 128, "target_length": 192, "grouping_target_length": 192, "note": "Original row IDs, not distinct question texts; row coverage does not prove untruncated token coverage."}
drive_checkpoint_verified=/content/drive/MyDrive/HSE_NLP_HW2/runs/activation-muon-2k-v1/control/step-002048
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
generated=1919 seconds=4.10 device=cuda:0 precision=torch.float32 invalid_before_retry=27 beam_repaired=0 handwritten_repairs=0
checkpoint_ready_for_bot=/content/drive/MyDrive/HSE_NLP_H

## Activation comparison completed — retain gated GELU

Both 42.48M models completed 2,048 updates from identical full random weights and consumed exactly the same 262,144 original rows, with zero repeats and zero completed passes. Strict greedy predictions and weights were saved to Drive and exported with matching hashes before bot scoring.

| Activation | Pilot /245 | Confirmation /493 | Execution failures | Public /960 |
|---|---:|---:|---:|---:|
| Gated GELU | 26 | 49 | 159 | 107 |
| SwiGLU | 20 | 53 | 191 | 108 |

SwiGLU fails the predeclared quality gate (-6/+4 correct, +32 failures). Keep gated GELU; no long run or second-seed quality confirmation is triggered. Logged throughput was 1,339.9 versus 1,701.9 examples/s, but this unequal-time test does not establish accuracy at matched compute. Driver time: 447.17 seconds; GPU utilization median90%, mean69.58%, peak18,681MiB. Allocation/authentication/install time is additional.

Worker exit0 and **zero active Colab sessions** verified. Best mature result remains575/960 (59.90%), rank10/16 at22:47(+03). [Saved run](https://drive.google.com/drive/folders/1Yk6CC2cZ2-5IiP8QAVSM3gFImLPYOfVQ). Source2735e1e. Final T4 timing and a trained full pass remain unverified.

The separate complete-sequence memory/throughput probe is prepared on CPU and byte-verified on Drive; it has not run. Its weights will be discarded, with no quality claim.

## Complete-sequence GPU probe — bounded, no retained weights

Run the CPU-prepared source a88585879408275e8060b64d851ca5b8b74e3a11 from Drive. Compare capped and complete original sequences in two reversed orders: three warmup and 32 measured updates each, plus 130 length-extreme examples. Same fresh 42.48M model and Muon settings. The worker has a 360-second bound and releases its runtime after saving results. This is a memory/throughput check, not an accuracy experiment; all weights are discarded. No full-pass claim is possible from this probe.

In [1]:
from google.colab import runtime
try:
 from google.colab import drive
 from pathlib import Path
 import hashlib
 drive.mount('/content/drive',timeout_ms=180000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/untruncated-gpu-probe-v1/launch.py')
 payload=p.read_bytes()
 assert hashlib.sha256(payload).hexdigest()=='76a84b5691330419d2189e181432b619d2581e9867b06657a7740935cb87dbaf'
 untruncated_probe_ns={}
 exec(compile(payload,str(p),'exec'),untruncated_probe_ns)
except BaseException:
 runtime.unassign()
 raise


Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
UNTRUNCATED_PROBE_STARTED 4108 hard bound 360 seconds; discard every probe weight


## Complete-sequence probe passed; GPU released

All four speed runs and 130 original long-example stress inputs completed with finite updates. Complete sequences averaged 1,162.7 examples/s versus 1,372.2 capped (18.0% more time), with 16.93 GiB peak allocated / 24.72 GiB reserved memory. The stress test peaked at 8.22 GiB. Median GPU activity was 95%. Measured driver time was 18.05s; worker lifetime 27.38s; allocation, authorization and installation are additional.

Results were exported from [Drive](https://drive.google.com/drive/folders/1-xMrSWvatcCUbeOX59WfAzj16MkkiQl9). Worker exit 0 and zero active sessions verified. No checkpoint retained, no quality improvement or full training pass claimed. Scaling remains on hold. A CPU-only check predicts fewer chunks with larger budgets (44 to 34 at 2x), but larger budgets are not GPU-tested yet.

## Bounded complete-sequence accuracy curve

Source d3c8136a68830bca45f26ab4fcad31450d344f81. Continue our best own scratch-trained 42.48M checkpoint at step73,024, preserving AdamW moments. Use complete original sequences, exact consumed-row/token accounting, batch128, BF16, numeric loss weight2, peak rate3e-5/warmup50. Retain steps73,728 /74,752 /75,776 /76,800: 3,776 added updates, 483,328 newly tracked original rows. Four strict-greedy outputs must be saved to Drive before export and bot scoring. No automatic longer run or full-pass claim.

26 CPU tests passed. Parent weights, optimizer moments and all60 data/index shards verified. Actual parent forward/backward is finite on original examples reaching question741/program1208tokens. The complete-sequence GPU memory gate already passed. Worker hard bound900seconds, followed by runtime release. Larger chunk budgets remain untested and are not used.

In [1]:
from google.colab import runtime
try:
 from google.colab import drive
 from pathlib import Path
 import hashlib
 drive.mount('/content/drive',timeout_ms=180000)
 p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/complete-curve-4k-v1/launch.py')
 payload=p.read_bytes()
 assert hashlib.sha256(payload).hexdigest()=='5893c82dc2b4ee1e0d65f94114e9199a8271d032908bc206ace1a5e4180a7c70'
 complete_curve_ns={}
 exec(compile(payload,str(p),'exec'),complete_curve_ns)
except BaseException:
 runtime.unassign()
 raise


Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
COMPLETE_CURVE_STARTED 3024 Hard bound 900 seconds; one 3776-update continuation, four retained checkpoints.


In [6]:
from pathlib import Path
import subprocess
print('WORKER',complete_curve_ns['full_process'].poll())
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,power.draw','--format=csv,noheader'],text=True))
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/runs/complete-curve-4k-v1/complete/training.log')
print(p.read_text()[-3000:] if p.exists() else complete_curve_ns['local_log'].read_text()[-3000:])


WORKER None
96 %, 34633 MiB, 362.33 W

38003], "excluded_by_shard": [1048, 26, 17, 17, 23, 23, 23, 24, 23, 24, 31, 25, 19, 23, 12, 8, 7, 6, 9, 6, 4, 2, 8, 4, 4, 7, 2, 12, 8, 1, 0, 0, 0, 0, 0, 0, 3, 10, 15, 5, 9, 10, 7, 9, 6, 5, 8, 2, 5, 5, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0], "unique_visited_by_shard": [5872, 5859, 5851, 5891, 5796, 5858, 5844, 5871, 5961, 5882, 5859, 5904, 5853, 5908, 5891, 5902, 5776, 5845, 5861, 5857, 5882, 5889, 5865, 5884, 5905, 5847, 5895, 5851, 5873, 5851, 5912, 5844, 5854, 5920, 5882, 5863, 5909, 5872, 5865, 5843, 5864, 5894, 5837, 5972, 5888, 5896, 5852, 5897, 5849, 5881, 5841, 5848, 5890, 5811, 5875, 5897, 5867, 5840, 5805, 5905], "presentations_by_shard": [5872, 5859, 5851, 5891, 5796, 5858, 5844, 5871, 5961, 5882, 5859, 5904, 5853, 5908, 5891, 5902, 5776, 5845, 5861, 5857, 5882, 5889, 5865, 5884, 5905, 5847, 5895, 5851, 5873, 5851, 5912, 5844, 5854, 5920, 5882, 5863, 5909, 5872, 5865, 5843, 5864, 5894, 5837, 5972, 5888, 5896, 5852, 5897, 5849, 5881, 5841, 5848, 5

## Complete-sequence curve finished — new public best, no long extension

Source d3c8136a68830bca45f26ab4fcad31450d344f81. All four checkpoints were saved to Drive, exported with verified model/output hashes, and scored by @hse_nlp_hw2_2026_bot.

| Step | Newly tracked unique rows | Pilot /245 | Confirmation /493 | Execution failures | Public /960 |
|---|---:|---:|---:|---:|---:|
| 73,728 | 90,112 | 139 | 285 | 38 | 574 |
| 74,752 | 221,184 | 136 | 295 | 42 | 581 |
| 75,776 | 352,256 | 139 | 291 | 43 | 583 |
| 76,800 | 483,328 | 140 | 288 | 43 | 569 |

Retain step75,776 as the new public best:583/960 (60.73%), eight answers above575/960. Latest rank10/16 at23:15(+03). Held-out gains are mixed; no automatic long run. Each checkpoint preserves complete original sequences with zero repeated tracked rows. No checkpoint has a completed full-pass receipt.

Driver512.01s (8.53min), including restoration, training, checkpoint exports and held-out evaluation. Training process408.97s. Logged throughput1,232.4 examples/s includes intermediate checkpoint work; whole-driver median GPU utilization94%, mean71.28%, peak34,633MiB. Allocation, Drive sign-in and installation are additional. Worker exit0 and zero active Colab sessions verified. [Saved run](https://drive.google.com/drive/folders/1M9kSWYEy8Tix1XE_sX9zFUROWbaWOXhK).

Inference notebook is pinned to the new best and statically checked; its preparation bundle has not run. Final T4 timing and first place remain unverified/unachieved.